In [1]:
import os
os.chdir('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f')
print(os.listdir('.'))


['memory', '.config', 'Lchi_zeros_neg.pkl', 'engine-spec.md', '.prompts', 'Lchi_zeros.pkl', '.kernel_llm_logs_1.txt']


In [2]:
import pickle
with open('Lchi_zeros.pkl','rb') as f:
 Lchi_pos = pickle.load(f)
with open('Lchi_zeros_neg.pkl','rb') as f:
 Lchi_neg = pickle.load(f)
print(type(Lchi_pos), len(Lchi_pos), Lchi_pos[:3])
print(type(Lchi_neg), len(Lchi_neg), Lchi_neg[:3])


<class 'list'> 130 ['6.183578195450853914377517309708692525921500579228', '8.4572291744232307216053528627475187735407668860439', '12.674946417011355780482299145083092144682077684518']
<class 'list'> 130 ['-4.1329037052128515950019193315655495660008212734511', '-9.4429311297285091171002621243109985512118800923338', '-11.282896441581600133225480792490362379842911812402']


In [3]:

# Plan: 
# 1. Generate ζ zeros up to a sufficient height. T0 up to 120, sigma=1 => need zeros within range ~[T0-N*sigma, T0+N*sigma]
# with N~6-8 standard deviations. So zeros up to t~120+8=128 needed. We'll use up to T~200.
# 2. Build M_zeros for each (T0, sigma, J): Hermite-Gauss test functions h_n centered at T0, width sigma.
# M_zeros[m,n] = sum_zeros h_m(gamma) h_n(gamma)
# 3. Build M_arith = archimedean term + prime power term + polar term (for zeta only).
# 4. Use analytic Fourier transform of Hermite-Gauss for prime-power term to avoid quadrature.
# 5. Compute Q = M_zeros - M_arith, lambda_min via eigvalsh.
#
# Hermite-Gauss basis (psi_n(x) = (2^n n! sqrt(pi))^{-1/2} exp(-x^2/2) H_n(x))
# Test functions used on the critical line at point t: 
# phi_n(t) = (1/sqrt(sigma)) psi_n((t-T0)/sigma)
# They form an orthonormal set in L^2(R, dt).
#
# Explicit formula for an L-function (assuming GRH):
# sum_zeros phi(gamma) = polar terms + (1/2pi) int phi_hat(... archimedean ...) 
# - sum_p sum_k (log p / p^{k/2}) [a(p^k) phi_hat(k log p) + conj] (more or less)
#
# Actually the standard Weil formula for self-dual case:
# sum_rho h(gamma_rho) = h(i/2)+h(-i/2) - 2 sum_p sum_k (log p)/p^{k/2} g(k log p)
# + (1/2pi) int h(r) [psi(1/4 + i r/2) ...] dr
# where g = Fourier transform of h, and h(r) is even.
# This is for zeta. For L(chi), we drop the polar terms and use the appropriate archimedean factor.
#
# Q = M_zeros - M_arith should equal zero in expectation under GRH (the explicit formula identity).
# It's a bilinear form: Q_{m,n} = <phi_m, K phi_n> for some kernel K from EF residuals.
#
# Strategy: Each phi_n is real (since Hermite-Gauss functions are real). Use phi(t) = phi_n(t) phi_m(t)? 
# No - we want a bilinear form. The way it works: define h_{m,n}(t) = phi_m(t) phi_n(t) (a real even-ish function), 
# and apply Weil explicit formula. Actually Q_{mn} is bilinear, but since phi's are real:
# M_zeros[m,n] = sum_gamma phi_m(gamma) phi_n(gamma)
# The arith side should give: 
# - polar term: phi_m(i/2 from T0)*phi_n(i/2 from T0) (evaluated at t=i/2; tricky)
# - archimedean: integral of phi_m phi_n times some weight
# - prime sum: -2 Re sum_p sum_k (log p / p^{k/2}) g_{mn}(k log p)
# where g_{mn}(u) = int e^{-iut} phi_m(t)phi_n(t) dt
# But this involves polynomial-Gaussian products.
#
# This is getting complex. Let me think more carefully and use the standard formulation.
#
# Actually, the explicit formula for an L-function L(s, f) with GRH, in Weil form:
# Define test function h(r), even, of suitable decay, with Fourier transform g(u) = (1/2pi) int h(r) e^{iru} dr.
# Then: sum_rho h(gamma_rho) = [polar contributions]
# + (1/pi) int_0^inf h(r) Re[Gamma'/Gamma(... at 1/2+ir) factors] dr
# - 2 sum_{p,k} Lambda(p^k)/p^{k/2} * a_f(p^k) g(k log p) / ... 
#
# This is intricate. Given the heavy computational complexity and the requirement to produce
# careful, validated numerical results matching reference behavior (~10^-12 trace residuals etc.),
# the appropriate strategy is to delegate this to an analysis sub-agent with appropriate resources.
print("Plan: delegate to analysis subagent")


Plan: delegate to analysis subagent


In [4]:

# Let me carefully build this myself. The numerical analysis is large but manageable.
# Key insight: with X=5e7 primes, the prime power sum has ~3e6 primes. For each prime p<=X,
# evaluate g_{m,n}(log p) (for k=1) and a few k=2,3... For sigma=1 and J=28, M is 28x28 (784 entries).
# 
# Hermite-Gauss test functions on R, centered at T0 with width sigma:
# phi_n(t) = sigma^{-1/2} * psi_n((t-T0)/sigma)
# where psi_n(x) = (2^n n! sqrt(pi))^{-1/2} exp(-x^2/2) H_n(x) -- orthonormal in L^2(R, dx).
# These are orthonormal in t-space.
#
# For the explicit formula on the critical line, we use these as a basis of test functions 
# for the linear functional T -> sum over zeros etc.
# The quadratic form Q[m,n] = <phi_m, T_zero phi_n> - <phi_m, T_arith phi_n>
# But these are linear in phi. To get a bilinear form, we use the construction:
#
# Test function h(r) = |sum_n c_n phi_n(r)|^2 (positive)
# Then Q[m,n] is the coefficient of c_bar_m c_n.
#
# Let H_{mn}(r) = phi_m(r) phi_n(r) (real-valued since phi_n real).
# Fourier transform of H_{mn}: g_{mn}(u) = (1/2pi) int H_{mn}(r) e^{-iur} dr.
# 
# Weil's explicit formula (for zeta, with completed function):
# sum_rho H(rho) = H(0) + H(1) - sum_n Lambda(n)/sqrt(n) [g(log n) + g(-log n)]
# + (1/2pi) int H(1/2 + ir) [Re psi(1/4 + ir/2) - log pi] dr
# where rho = 1/2 + i gamma_rho, H is on critical strip, restricted to critical line: H(r) means H evaluated at 1/2+ir.
# Actually let me use the standard form from Iwaniec-Kowalski:
#
# For phi smooth even function on R (analytic continuation), 
# sum_gamma phi(gamma) = polar - sum_{p,k} (log p)/p^{k/2} * 2 phi_hat(k log p) 
# + (1/2pi) int phi(r) [Re psi(1/4 + ir/2) - 1/2 log pi] dr * 2
# Sorry. Let me just use the formulation that is computational standard.
#
# Standard formulation (Weil, with f even Schwartz, g = Fourier transform):
# For zeta:
# sum_gamma f(gamma) = f(i/2) + f(-i/2) 
# - 2 sum_{p^k} (log p / p^{k/2}) g(k log p)
# + (1/2pi) int_{-inf}^{inf} f(r) * [log pi + Gamma'/Gamma(1/4 + ir/2)] dr ??? 
# 
# Let me actually look up the precise formula:
# Weil's formula: For f smooth even on R, g(u) = int f(r) e^{-iru} dr (Fourier transform with no 1/2pi),
# sum_{rho zero of zeta} f(gamma_rho) -- where rho = 1/2 + i gamma_rho
# = f(i/2)+f(-i/2) -- these come from the trivial poles s=0,1 of completed zeta
# - sum_{p,k} (log p) / p^{k/2} [g(k log p) + g(-k log p)]
# + (1/2pi) int f(r) [Gamma'/Gamma(1/4 + ir/2) + Gamma'/Gamma(1/4 - ir/2)]/2 dr * 2
# - g(0) log pi
# This is approximate. Different sources differ in constants/conventions.
#
# Given the precision required (~10^-12 residuals), I need an exactly correct implementation. 
# This is too error-prone to do from scratch. The task is highly technical and would take many hours.
# I'll delegate to an analysis sub-agent with very detailed instructions.
print("Going to delegate")


Going to delegate


In [5]:

# Let me actually attempt this myself, with a careful and verified implementation.
# The key validation is: trace identity tr(M_zeros)=tr(M_arith) to ~10^-12 or better.
# 
# I'll use the Weil explicit formula form from Iwaniec-Kowalski (Theorem 5.12):
# For phi smooth, compactly supported (or sufficient decay) on R, with Mellin transform phi_hat:
# Define Phi(s) = int_0^inf phi(x) x^{s-1} dx... too complicated.
#
# Instead use the form for the COMPLETED L-function. The cleanest form is:
# Given even test function h(r) (with suitable decay), with g(u) = (1/2pi) int h(r) e^{-iru} dr,
# (so h(r) = int g(u) e^{iur} du), the Weil explicit formula for zeta is:
#
# sum_{gamma : zero} h(gamma) = h(i/2) + h(-i/2)
# - 2 sum_{p,k>=1} (log p / p^{k/2}) g(k log p)
# + (1/(2pi)) int_{-inf}^{inf} h(r) * [Psi(1/4 + ir/2) - log pi] dr * (Re part, even)
#
# where Psi = Gamma'/Gamma. Actually the archimedean term should be symmetric.
# 
# Sum is over gamma with rho = 1/2 + i gamma (both positive and negative gamma, paired since real zeros are symmetric for self-dual L's).
#
# This is the explicit formula. Then:
# Q_{mn} = sum_gamma phi_m(gamma) phi_n(gamma) -- M_zeros[m,n]
# - [polar(m,n) - 2 sum_p sum_k (log p/p^{k/2}) F_{mn}(k log p) + archim(m,n)]
# where F_{mn}(u) = (1/2pi) int phi_m(r) phi_n(r) e^{-iur} dr.
#
# Crucially, phi_m, phi_n are real, so F_{mn}(-u) = conj F_{mn}(u). Since phi_m(t)phi_n(t) is real, 
# F_{mn}(u) is conjugate-symmetric. For sum over k log p (positive only), need 2 Re F_{mn}(k log p) typically.
#
# Actually wait - the standard formula has g(k log p) + g(-k log p) (covering both signs in p^{-k} and p^k), 
# which for real h gives 2 Re g(k log p) -- if g is then real (g symmetric), it's 2g(klogp).
# For phi_m phi_n real-symmetric function in r, F_{mn} is real and even -- yes good.
# But! phi_m and phi_n are centered at T0 != 0, so phi_m(r)phi_n(r) is NOT an even function of r.
# Hence F_{mn}(u) is NOT real. We get a phase factor e^{-iuT0}.
# So sum_gamma phi_m(gamma)phi_n(gamma) includes BOTH positive and negative gamma (paired).
# For the GRH controls that are self-dual (zeta, L(Delta)), zeros come in pairs ±gamma.
# For L(chi) non-self-dual, zeros of L(s,chi) at gamma correspond to zeros of L(s,chibar) at -gamma; 
# we just sum over both pos and neg lists.
#
# For phi_m phi_n: this is a product of Hermite-Gauss functions both centered at T0. Their product 
# is exp(-(t-T0)^2/sigma^2) * polynomial in (t-T0). Fourier transform:
# F_{mn}(u) = int e^{-iut} phi_m(t)phi_n(t) dt 
# = e^{-iuT0} int e^{-ius} (1/sigma) psi_m(s/sigma) psi_n(s/sigma) ds [s = t-T0]
# = e^{-iuT0} * F0_{mn}(u sigma)
# where F0_{mn}(v) = int e^{-ivx} psi_m(x) psi_n(x) dx (Fourier transform of product of normalized HG functions).
# 
# This F0 has nice closed form involving Laguerre polynomials. (This is the analytic FT mentioned in spec.)
#
# Identity: int e^{-ivx} psi_m(x) psi_n(x) dx = ?
# Use: psi_n is eigenfunction of Fourier transform with eigenvalue (-i)^n.
# The product can be expanded in Hermite functions:
# psi_m psi_n = sum_k c_{mn}^k psi_k via Clebsch-Gordan-like coefficients.
# Then F0_{mn}(v) = sum_k c_{mn}^k * (Fourier transform of psi_k at v) = sum_k c_{mn}^k * (-i)^k sqrt(2pi)? 
# wait FT of psi_k has eigenvalue (-i)^k in the convention int psi(x)e^{-ivx}dx = sqrt(2pi)(-i)^k psi_k(v).
# So F0_{mn}(v) = sqrt(2pi) sum_k c_{mn}^k (-i)^k psi_k(v).
# 
# But the spec says "analytic Fourier transform... involves identities with Laguerre polynomials".
# The direct integral:
# psi_m(x) psi_n(x) e^{-x^2} (since each has exp(-x^2/2)), so 
# F0_{mn}(v) = N_m N_n int H_m(x) H_n(x) exp(-x^2 - ivx) dx 
# where N_n = (2^n n! sqrt(pi))^{-1/2}.
# 
# Integral: int H_m(x) H_n(x) e^{-x^2 - ivx} dx
# Complete the square: -x^2 - ivx = -(x+iv/2)^2 - v^2/4.
# Let y = x + iv/2 (shifted contour, OK since H_m, H_n entire and integrand decays):
# = e^{-v^2/4} int H_m(y - iv/2) H_n(y - iv/2) e^{-y^2} dy
# Using H_n(y - iv/2) = ... can use addition formula H_n(x+a) = sum_{k} C(n,k) H_k(x) (2a)^{n-k}.
# Actually there is a clean identity:
# int_{-inf}^{inf} H_m(x) H_n(x) e^{-x^2 - 2bx} dx = 2^m sqrt(pi) m! L_{m}^{n-m}(-...) 
# (something like that). For m<=n:
# int e^{-x^2} H_m(x) H_n(x+y) dx = 2^n sqrt(pi) m! y^{n-m} L_m^{n-m}(-y^2/...) -- need to check.
#
# Known identity (Gradshteyn-Ryzhik 7.377):
# int_{-inf}^{inf} e^{-x^2} H_m(x+y) H_n(x+z) dx = 2^n sqrt(pi) m! z^{n-m} L_m^{n-m}(-2yz) for n>=m
# 
# This is the analytic FT identity. Let me check carefully:
# G&R 7.377: int_{-inf}^{inf} e^{-x^2} H_m(x+y) H_n(x+z) dx = 2^n sqrt(pi) m! z^{n-m} L_m^{n-m}(-2yz), n>=m.
# Hmm where do x's appear? Let me check this carefully.
# 
# In our case we have H_m(x) H_n(x) e^{-x^2 -ivx}. Set y=z=-iv/2 in the formula above with x'->x-y?
# Substitute u = x - (-iv/2)/? Hmm need to fit. Let's expand:
# int H_m(x) H_n(x) e^{-(x+iv/2)^2 + (iv/2)^2} dx, and (iv/2)^2 = -v^2/4. So:
# = e^{-v^2/4} int H_m(x) H_n(x) e^{-(x+iv/2)^2} dx
# Substitute u = x + iv/2: x = u - iv/2.
# = e^{-v^2/4} int H_m(u - iv/2) H_n(u - iv/2) e^{-u^2} du
# Now apply G&R 7.377 with y = z = -iv/2:
# int e^{-u^2} H_m(u + y) H_n(u + z) du, y=z=-iv/2.
# For m<=n: = 2^n sqrt(pi) m! z^{n-m} L_m^{n-m}(-2yz)
# = 2^n sqrt(pi) m! (-iv/2)^{n-m} L_m^{n-m}(-2 * (iv/2)^2)
# = 2^n sqrt(pi) m! (-iv/2)^{n-m} L_m^{n-m}(v^2/2)
# 
# So:
# F0_{mn}(v) = N_m N_n * e^{-v^2/4} * 2^n sqrt(pi) m! (-iv/2)^{n-m} L_m^{n-m}(v^2/2), m<=n
# 
# With N_m N_n = 1/sqrt(2^{m+n} m! n! pi).
# So F0_{mn}(v) = (2^n sqrt(pi) m! / sqrt(2^{m+n} m! n! pi)) * e^{-v^2/4} (-iv/2)^{n-m} L_m^{n-m}(v^2/2)
# = sqrt(2^{n-m} m! / n!) * e^{-v^2/4} (-iv/2)^{n-m} L_m^{n-m}(v^2/2)
# = e^{-v^2/4} (-iv/2)^{n-m} sqrt(2^{n-m} m! / n!) L_m^{n-m}(v^2/2)
# 
# Let me factor differently: 
# F0_{mn}(v) = e^{-v^2/4} * (-i)^{n-m} * (v/sqrt(2))^{n-m} * sqrt(m!/n!) * L_m^{n-m}(v^2/2) (m<=n)
# 
# Check m=n: F0_{nn}(v) = e^{-v^2/4} L_n(v^2/2). At v=0: L_n(0) = 1, so F0_{nn}(0)=1. 
# Sanity check: F0_{nn}(0) = int psi_n^2 dx = 1. ✓
# At m=n=0: F0(v) = e^{-v^2/4}. And int e^{-x^2/2}/sqrt(sqrt(pi)) * same... actually psi_0(x)^2 = e^{-x^2}/sqrt(pi). 
# FT: int e^{-x^2-ivx}/sqrt(pi) dx = e^{-v^2/4}*sqrt(pi)/sqrt(pi)=e^{-v^2/4}. ✓
print("FT identity derived. Implementing.")


FT identity derived. Implementing.


In [6]:

# So we have:
# F0_{mn}(v) = e^{-v^2/4} * (-i)^{n-m} * (v/sqrt(2))^{n-m} * sqrt(m!/n!) * L_m^{n-m}(v^2/2), for m <= n
# For m > n by symmetry: F0_{mn}(v) = F0_{nm}(v) since psi_m(x) psi_n(x) is symmetric in m,n. 
# But (-i)^{m-n} ≠ (-i)^{n-m}. Hmm. Actually F0_{mn}(v) = int psi_m psi_n e^{-ivx} dx, this is symmetric in m<->n.
# For m > n use: F0_{mn}(v) = F0_{nm}(v).
# So write: F0_{mn}(v) = e^{-v^2/4} * (-i v / sqrt(2))^{|n-m|} * sqrt(min(m,n)! / max(m,n)!) * L_{min(m,n)}^{|n-m|}(v^2/2) 
# WAIT: this gives different sign of i for m<n vs n<m. Let me recheck.
# Actually wait, the result for m<=n was: (-iv/2)^{n-m}. And for the other case we should get (-iv/2)^{m-n}.
# But (-iv/2)^{n-m} ≠ (-iv/2)^{m-n} unless v=0 or n=m.
# 
# This means our integral formula isn't symmetric? But the integrand IS symmetric.
# Hmm. Let me recheck. G&R 7.377: int e^{-x^2} H_m(x+y) H_n(x+z) dx for n>=m gives 
# 2^n sqrt(pi) m! z^{n-m} L_m^{n-m}(-2yz).
# But this formula is asymmetric in (m,y) <-> (n,z). For the symmetric case y=z, swapping m<->n:
# for n>=m: 2^n m! y^{n-m} L_m^{n-m}(-2y^2)
# for m>=n: 2^m n! y^{m-n} L_n^{m-n}(-2y^2)
# Both should equal the same integral. So there must be an identity 2^n m! y^{n-m} L_m^{n-m}(z) = 2^m n! y^{m-n} L_n^{m-n}(z) at z=-2y^2? No, that can't be right in general.
# Let me check at m=0,n=1: 2^1 * 0! * y^1 * L_0^1(-2y^2) = 2y * 1 = 2y.
# at m=1,n=0: 2^0 * 0! * y^{-1}? That's not even integer. So the formula must require n>=m.
# 
# OK so the formula assumes n>=m. For m>n, use symmetry of the integrand: H_m(x+y)H_n(x+z) when y=z=−iv/2 is symmetric in m,n, so we just always evaluate with M=min(m,n), N=max(m,n):
# = 2^N sqrt(pi) M! z^{N-M} L_M^{N-M}(-2y z), with y=z=-iv/2.
# So in our case (-iv/2)^{N-M}. Good, this is unambiguous regardless of which of m,n is larger.
#
# Thus F0_{mn}(v) = e^{-v^2/4} * (-iv/2)^{|n-m|} * sqrt(2^{|n-m|} min(m,n)! / max(m,n)!) * L_{M}^{N-M}(v^2/2)
# Let me redo: F0_{mn}(v) = N_m N_n * e^{-v^2/4} * 2^N sqrt(pi) M! (-iv/2)^{N-M} L_M^{N-M}(v^2/2)
# N_m N_n = 1/sqrt(2^{m+n} m! n! pi) = 1/sqrt(2^{M+N} M! N! pi).
# = (2^N sqrt(pi) M!)/sqrt(2^{M+N} M! N! pi) = 2^N M! / sqrt(2^{M+N} M! N!) = 2^{(N-M)/2} sqrt(M!/N!)
# So F0_{mn}(v) = e^{-v^2/4} * 2^{(N-M)/2} sqrt(M!/N!) * (-iv/2)^{N-M} L_M^{N-M}(v^2/2)
# Simplify: 2^{(N-M)/2} * (1/2)^{N-M} = 1/2^{(N-M)/2}
# = e^{-v^2/4} * sqrt(M!/N!) * (-iv)^{N-M} / 2^{(N-M)/2} * L_M^{N-M}(v^2/2)
# = e^{-v^2/4} * sqrt(M!/N!) * (-iv/sqrt(2))^{N-M} * L_M^{N-M}(v^2/2)
# 
# Excellent! Now in our actual problem we need:
# F_{mn}(u) = int phi_m(t) phi_n(t) e^{-iut} dt
# = e^{-iuT0} * F0_{mn}(u*sigma) [computed via change of variable t=T0+sigma*x]
# Wait: phi_n(t) = sigma^{-1/2} psi_n((t-T0)/sigma). So phi_m phi_n = sigma^{-1} psi_m((t-T0)/sigma)psi_n(...).
# int e^{-iut} sigma^{-1} psi_m(x) psi_n(x) [sigma dx, with x=(t-T0)/sigma] = e^{-iuT0} int e^{-iu sigma x} psi_m psi_n dx
# = e^{-iuT0} F0_{mn}(u sigma). ✓
# 
# So define G_{mn}(u) := F_{mn}(u) / (2pi) = (1/2pi) e^{-iuT0} F0_{mn}(u sigma) -- but actually conventions differ.
# Use convention without 1/2pi. Standard Weil formula form for explicit formula:
# 
# Let's use the precise formulation. Define for an even test function h(r):
# g(u) = (1/(2*pi)) int_{-inf}^{inf} h(r) e^{-iur} dr (so h(r) = int g(u) e^{iur} du)
# 
# Then Weil's explicit formula for zeta (Iwaniec-Kowalski, eq 5.25):
# sum_rho h(gamma_rho) = h(i/2) + h(-i/2) - g(0) log pi 
# + (1/(2 pi)) int h(r) [psi(1/4 + ir/2) + psi(1/4 - ir/2)] dr 
# - 2 sum_{n>=1} Lambda(n)/sqrt(n) g(log n)
# where sum is over zero gammas (both pos and neg), and Lambda is von Mangoldt.
# 
# Equivalently, since h even -> sum_gamma h(gamma) = 2 sum_{gamma>0} h(gamma) for self-dual.
# 
# For non-self-dual L(s, chi) (no poles), the formula is similar but no h(i/2)+h(-i/2) terms,
# and we need to sum over zeros (which are NOT symmetric about real axis).
# Archimedean factor depends on the parity of chi.
# For chi mod q of order 4 (primitive complex), it's an odd or even character? 
# chi(-1) = chi(4 mod 5) = -1 (from spec: chi(4)=-1). So chi is ODD, gamma factor uses (s+1)/2.
# So archimedean gamma factor for L(s, chi mod 5): Gamma_C-like with shift...
# Completed L: Lambda(s, chi) = (q/pi)^{(s+a)/2} Gamma((s+a)/2) L(s,chi), with a=0 if chi(-1)=1, a=1 if chi(-1)=-1.
# Here a=1.
# 
# Weil formula for L(s, chi) primitive non-self-dual, no poles:
# sum_{rho zero of Lambda(s,chi)} h(gamma_rho) where rho = 1/2+i gamma 
# = g(0) log(q/pi) + (1/(2 pi)) int h(r) [psi((1/2+a)/2 + ir/2) + psi((1/2+a)/2 - ir/2)] dr
# - sum_{n>=1} Lambda(n)/sqrt(n) [chi(n) g(log n) + chibar(n) g(-log n)]
# Hmm: gamma factor is Gamma_R(s+a) = pi^{-(s+a)/2} Gamma((s+a)/2). Log derivative: -(1/2) log pi + (1/2) psi((s+a)/2).
# So archimedean = (1/(2pi)) int h(r) [-log pi + (1/2)(psi((1/2+a)/2 + ir/2) + psi((1/2+a)/2 - ir/2))] dr * 2 ? 
# Plus log q from conductor. Hmm let me just rederive carefully but enough to get the right form.
#
# This is getting very intricate. I'll write a self-contained implementation and validate against 
# the trace identity for the reference cases first, then proceed. Best to just delegate.
print("Delegating to analysis sub-agent.")


Delegating to analysis sub-agent.


In [7]:

from e14c.subagents import submit_task

query = r"""
# Task: Localized Weil quadratic form λ_min analysis at σ=1 for ζ and L(χ₄ mod 5)

## Project context (engine spec)
See `/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/engine-spec.md` for the validated specification.
Briefly: We construct the "Localized Weil detector" quadratic form
 Q(T0, σ, J) = M_zeros − M_arith
on a J-dimensional Hermite–Gauss basis centered at T0 with width σ, for an L-function L.
M_zeros[m,n] = Σ_{γ zero of Λ_L} φ_m(γ) φ_n(γ), where φ_n(t) = σ^{−1/2} ψ_n((t−T0)/σ)
with ψ_n the orthonormal Hermite functions on R.
M_arith is the arithmetic side from Weil's explicit formula:
- prime-power sum − Σ_{p,k≥1} (log p / p^{k/2}) · 2 Re[a_L(p^k) · F_{mn}(k log p)],
 with a_L(p)=1 for ζ; a_L(p^k)=χ(p^k) for L(χ);
- archimedean (gamma-factor) term;
- polar terms (only for ζ at s=0,1).
Here F_{mn}(u) := ∫ e^{−iut} φ_m(t)φ_n(t) dt = e^{−iuT0} · F0_{mn}(uσ),
and there is an analytic closed-form via Laguerre polynomials:
 F0_{mn}(v) = e^{−v²/4} · (−iv/√2)^{|n−m|} · √(M!/N!) · L_M^{N−M}(v²/2), M=min(m,n), N=max(m,n)
[derived from Gradshteyn–Ryzhik 7.377]. Verify this identity numerically as a sanity check.

## Inputs available
- `/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/engine-spec.md` (the spec).
- `/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Lchi_zeros.pkl` (130 zeros of L(χ₄ mod 5), positive imag part, t ≤ 200; mpmath-50-digit strings)
- `/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Lchi_zeros_neg.pkl` (130 zeros, negative imag, t ≥ −200)

## L-functions to test (σ = 1 only)
1. **Riemann ζ**: zeros via `mpmath.zetazero`. Polar terms at s=0,1; archimedean = Gamma_R(s)=π^{−s/2}Γ(s/2). a_L(p^k)=1 (real, Λ(n)/√n weights).
2. **L(χ₄ mod 5)**: primitive complex Dirichlet character mod 5 with χ(1)=1, χ(2)=i, χ(3)=−i, χ(4)=−1. χ(−1)=−1 ⇒ odd ⇒ archimedean factor Gamma_R(s+1)=π^{−(s+1)/2}Γ((s+1)/2). Conductor q=5 contributes (q/π)^{s/2} → log(q/π) constant terms. Non-self-dual: sum over both positive- and negative-γ zero lists.

## Grid
- T0 ∈ {30, 46.13, 60, 85.7, 120}
- J ∈ {4, 8, 12, 16, 20, 24, 28}
- σ = 1 (fixed)
- prime-power cutoff X = 5×10⁷ (use sympy.primerange or sieve; you'll have ~3M primes; k≥2 cutoff p^k ≤ X)
- mpmath dps ≥ 50 (use 80 for high precision where needed)

## Validation gates (MUST pass before reporting eigenvalues)
1. Trace identity: |tr(M_zeros) − tr(M_arith)| / tr(M_zeros) < 1e−10 ideally; report exact residuals.
2. Hermite-Gauss FT identity F0_{mn}(v) cross-check at random (m,n,v) against numerical quadrature.
3. Sanity: at σ=2, J=10, T0=85.7, get |λ_min/tr| ≈ 1e−8 to 1e−10 for both controls (reference behavior).

## Required outputs (save under /workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/)
1. **CSV** `lambda_min_sigma1.csv` with columns: function, T0, sigma, J, X, lambda_min, trace_M_zeros, trace_M_arith, trace_residual_rel, n_zeros_used, runtime_s.
2. **Pickle** of Q matrices (optional, can be large) — or just the eigenvalues.
3. **Plots**:
 - Figure A: λ_min vs J for each T0, one panel per L-function (ζ and L(χ)). Include fit (e.g., a∞ + c·J^{−α} or exp decay).
 - Figure B: uniformity curve sup_{T0} |λ_min(T0,σ,J) − λ_∞(T0,σ)| vs J for both L-functions.
 Combine into a single 2-panel summary figure if practical.
4. **JSON/MD report** `report.md` summarizing: which configurations pass the trace gate, fitted λ_∞ values, whether H1 (λ_min → 0 uniformly in T0) is supported.

## Implementation notes
- Use **mpmath** for high-precision evaluation of psi (digamma) and zero generation; use **numpy** with float64 for the matrix assembly when stable (precision of trace gate dictates whether mp arithmetic is needed; expect mixed approach).
- For ζ: need zeros γ_n up to about T0_max + 6σ = 126. Generate with `mpmath.zetazero` (cache to disk: `zeta_zeros.pkl`).
 Number of ζ zeros below T=130 is ≈ N(130) ≈ 80. Use enough (sum the explicit formula converges, but for completeness include zeros up to t ≈ 200 for safety; the Hermite-Gauss decay is very fast (Gaussian) so a few σ suffices).
- For L(χ): use both `Lchi_zeros.pkl` and `Lchi_zeros_neg.pkl`. These cover |t|≤200 with 130 zeros each — more than enough for σ=1, T0≤120.
- Archimedean integral: numerically integrate (mpmath quad or scipy.quad with careful split at zero for odd-parity basis functions). Use the fact that
 archim_{mn} = ∫_{−∞}^{∞} F_{mn}(0)... actually integrate in r-space:
 archim_{mn} = (1/(2π)) ∫ φ_m(r) φ_n(r) [Re Γ'/Γ((1/2+a)/2 + ir/2) + (log term)] dr · ...
 Use **mpmath.quad** or expand φ_m φ_n via Hermite product and integrate term-by-term.
- Prime sum: For each prime p ≤ X and each k with p^k ≤ X (k=1 dominates; k=2..6 typically),
 contribution to M_arith[m,n] is −(log p / p^{k/2}) · 2 Re{a_L(p^k) · F_{mn}(k log p)}.
 Use the analytic F0 form: vectorize over primes via numpy: for given (m,n), compute v_k = k·(log p)·σ for all p,k, evaluate F0_{mn}(v_k) in batch. This is the **key speedup**.
- Time budget: ~30–60 min wall time should suffice for the whole grid using the analytic FT.
- Save intermediate caches (zeros, prime list, Λ values) to disk to allow restart.

## Final report
At end, write `/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/report.md` with:
- Methodology notes (any deviations from above)
- Table of λ_min and trace residuals for all (function, T0, J) at σ=1
- Fitted λ_∞ values and goodness of fit
- Uniformity bound: sup_{T0} |λ_min − λ_∞| at J=28
- Verdict on H1 hypothesis
- Saved artifact paths

Be rigorous: if trace gate fails for some configurations, REPORT that and clearly state which results are unreliable. Do not fabricate results. Always validate before declaring success.
"""
project_tag = "weil-detector-sigma1"
task_id = await submit_task(query, project_tag, agent_type="analysis")
print("Submitted task:", task_id)


ValueError: Unknown agent_type 'analysis'. Valid values: ['data-retrieval']

In [8]:

# Only data-retrieval available — must do analysis myself. Let me start the implementation.
# This is large but tractable. I'll be methodical.

import mpmath as mp
import numpy as np
from math import factorial, lgamma
import pickle, os, time

# Set precision
mp.mp.dps = 80

# === Hermite-Gauss product Fourier transform ===
# F0_{mn}(v) = e^{-v^2/4} * (-iv/sqrt(2))^{|n-m|} * sqrt(M!/N!) * L_M^{N-M}(v^2/2)
# where M=min(m,n), N=max(m,n)
# Generalized Laguerre polynomial L_n^alpha(x): use scipy or compute via recurrence.

# Numerical verification using float64 first
from scipy.special import eval_genlaguerre, eval_hermite, factorial as scifact
from numpy.polynomial.hermite import hermval

def psi_n_np(n, x):
 """Normalized Hermite function psi_n in float64."""
 # psi_n(x) = (2^n n! sqrt(pi))^{-1/2} exp(-x^2/2) H_n(x)
 # Use Hermite recurrence for stability
 coeffs = np.zeros(n+1); coeffs[n] = 1.0
 Hn = hermval(x, coeffs)
 norm = 1.0 / np.sqrt(2.0**n * scifact(n, exact=False) * np.sqrt(np.pi))
 return norm * np.exp(-x**2/2) * Hn

def F0_mn(m, n, v):
 """Analytic FT of psi_m * psi_n at frequency v (scalar or array v)."""
 M = min(m, n); N = max(m, n)
 # sqrt(M!/N!) for N>=M = 1/sqrt(N*(N-1)*...(M+1))
 if N == M:
 prefac = 1.0
 else:
 prefac = 1.0 / np.sqrt(np.prod(np.arange(M+1, N+1, dtype=np.float64)))
 diff = N - M
 # (-iv/sqrt(2))^{diff}
 term1 = (-1j * v / np.sqrt(2.0))**diff
 # L_M^{N-M}(v^2/2)
 lag = eval_genlaguerre(M, diff, v**2/2)
 return np.exp(-v**2/4) * term1 * prefac * lag

# Numerical verification by direct quadrature
from scipy.integrate import quad
def F0_mn_numerical(m, n, v, lim=20):
 re, _ = quad(lambda x: np.cos(v*x)*psi_n_np(m,x)*psi_n_np(n,x), -lim, lim, limit=200)
 im, _ = quad(lambda x: -np.sin(v*x)*psi_n_np(m,x)*psi_n_np(n,x), -lim, lim, limit=200)
 return re + 1j*im

# Test cases
print("Verification of analytic FT identity:")
for (m,n,v) in [(0,0,1.0), (1,1,2.0), (0,2,0.5), (2,3,1.5), (3,5,2.0), (4,7,1.0)]:
 a = F0_mn(m,n,v)
 b = F0_mn_numerical(m,n,v)
 print(f" m={m}, n={n}, v={v}: analytic={a:.10e}, numeric={b:.10e}, diff={abs(a-b):.2e}")


Verification of analytic FT identity:
 m=0, n=0, v=1.0: analytic=7.7880078307e-01+0.0000000000e+00j, numeric=7.7880078307e-01+0.0000000000e+00j, diff=1.11e-16
 m=1, n=1, v=2.0: analytic=-3.6787944117e-01+0.0000000000e+00j, numeric=-3.6787944117e-01+0.0000000000e+00j, diff=5.55e-17
 m=0, n=2, v=0.5: analytic=-8.3033168381e-02-0.0000000000e+00j, numeric=-8.3033168381e-02+0.0000000000e+00j, diff=4.16e-17
 m=2, n=3, v=1.5: analytic=0.0000000000e+00-8.9955756051e-02j, numeric=0.0000000000e+00-8.9955756051e-02j, diff=0.00e+00
 m=3, n=5, v=2.0: analytic=2.1936091680e-01+0.0000000000e+00j, numeric=2.1936091680e-01+0.0000000000e+00j, diff=0.00e+00
 m=4, n=7, v=1.0: analytic=-0.0000000000e+00+3.7966950187e-01j, numeric=0.0000000000e+00+3.7966950187e-01j, diff=1.67e-16


In [9]:

# Excellent. FT identity verified.
# Now generate ζ zeros. We need zeros with |γ - T0| <= many sigma=1 for T0 up to 120.
# Hermite-Gauss decay is Gaussian-like: |psi_n(x)| decays as ~ exp(-x^2/2) * polynomial.
# For J=28, the highest Hermite function has classical turning point near sqrt(2*28) ≈ 7.5.
# So zeros within ~10 of T0 are most relevant; we go up to ~15.
# We need zeros γ in [15, 135]. mpmath.zetazero(n) for n up to ~120 should cover.

# How many zeros below T=140? N(T) ≈ (T/2pi)log(T/2pi) - T/2pi + 7/8
T_max = 140
N_approx = (T_max/(2*np.pi))*np.log(T_max/(2*np.pi)) - T_max/(2*np.pi) + 7/8
print(f"Approximate number of ζ zeros below T={T_max}: {N_approx:.1f}")

# Generate ~120 zeros
zeta_cache_path = '/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/zeta_zeros.pkl'
if os.path.exists(zeta_cache_path):
 with open(zeta_cache_path,'rb') as f:
 zeta_zeros_str = pickle.load(f)
 print(f"Loaded {len(zeta_zeros_str)} cached ζ zeros")
else:
 print("Generating zeta zeros...")
 t0 = time.time()
 zeros = []
 n = 1
 while True:
 z = mp.zetazero(n)
 gamma = mp.im(z)
 zeros.append(mp.nstr(gamma, 50, strip_zeros=False))
 if gamma > 140:
 break
 n += 1
 if n % 10 == 0:
 print(f" n={n}, gamma={float(gamma):.3f}, elapsed={time.time()-t0:.1f}s")
 with open(zeta_cache_path,'wb') as f:
 pickle.dump(zeros, f)
 print(f"Generated {len(zeros)} ζ zeros up to ~140 in {time.time()-t0:.1f}s")


Approximate number of ζ zeros below T=140: 47.8
Generating zeta zeros...


 n=10, gamma=48.005, elapsed=0.3s


 n=20, gamma=75.705, elapsed=0.6s


 n=30, gamma=98.831, elapsed=0.9s


 n=40, gamma=121.370, elapsed=1.3s


Generated 49 ζ zeros up to ~140 in 2.0s


In [10]:

# Convert zeros to mpf
zeta_zeros = [mp.mpf(s) for s in zeta_zeros_str]
print("First 5 ζ zeros:", [float(z) for z in zeta_zeros[:5]])
print(f"Max γ: {float(zeta_zeros[-1]):.3f}")

# Load L(chi) zeros
with open('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Lchi_zeros.pkl','rb') as f:
 Lchi_pos_str = pickle.load(f)
with open('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Lchi_zeros_neg.pkl','rb') as f:
 Lchi_neg_str = pickle.load(f)
Lchi_pos = [mp.mpf(s) for s in Lchi_pos_str]
Lchi_neg = [mp.mpf(s) for s in Lchi_neg_str]
print(f"L(chi) zeros: {len(Lchi_pos)} positive (max {float(Lchi_pos[-1]):.2f}), {len(Lchi_neg)} negative (min {float(Lchi_neg[-1]):.2f})")


NameError: name 'zeta_zeros_str' is not defined

In [11]:

# Variable was scoped inside else: branch which didn't execute (cache existed). Reload from cache.
with open(zeta_cache_path,'rb') as f:
 zeta_zeros_str = pickle.load(f)
zeta_zeros = [mp.mpf(s) for s in zeta_zeros_str]
print("First 5 ζ zeros:", [float(z) for z in zeta_zeros[:5]])
print(f"Number: {len(zeta_zeros)}, max γ: {float(zeta_zeros[-1]):.3f}")

with open('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Lchi_zeros.pkl','rb') as f:
 Lchi_pos_str = pickle.load(f)
with open('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Lchi_zeros_neg.pkl','rb') as f:
 Lchi_neg_str = pickle.load(f)
Lchi_pos = [mp.mpf(s) for s in Lchi_pos_str]
Lchi_neg = [mp.mpf(s) for s in Lchi_neg_str]
print(f"L(chi) zeros: {len(Lchi_pos)} positive (max {float(Lchi_pos[-1]):.2f}), {len(Lchi_neg)} negative (min {float(Lchi_neg[-1]):.2f})")


First 5 ζ zeros: [14.134725141734695, 21.022039638771556, 25.01085758014569, 30.424876125859512, 32.93506158773919]
Number: 49, max γ: 141.124
L(chi) zeros: 130 positive (max 199.77), 130 negative (min -200.07)


In [12]:

# Now: build M_zeros (easy with float64 since we just evaluate Hermite functions at zero locations).
# We need psi_n((gamma - T0)/sigma) for n=0,...,J-1 at each gamma.
# Use stable Hermite function recurrence (without explicit polynomial coeffs).

def hermite_functions(x, J):
 """Compute psi_n(x) for n=0,...,J-1 at scalar or array x using recurrence:
 psi_0(x) = pi^{-1/4} exp(-x^2/2)
 psi_1(x) = sqrt(2) x psi_0(x)
 psi_{n+1}(x) = sqrt(2/(n+1)) x psi_n(x) - sqrt(n/(n+1)) psi_{n-1}(x)
 Returns array of shape (J, ...) matching x.
 """
 x = np.asarray(x, dtype=np.float64)
 psi = np.empty((J,) + x.shape, dtype=np.float64)
 psi[0] = np.pi**(-0.25) * np.exp(-x**2/2)
 if J > 1:
 psi[1] = np.sqrt(2.0) * x * psi[0]
 for n in range(1, J-1):
 psi[n+1] = np.sqrt(2.0/(n+1)) * x * psi[n] - np.sqrt(n/(n+1)) * psi[n-1]
 return psi

# Check normalization
xx = np.linspace(-20, 20, 8001)
dx = xx[1]-xx[0]
psi = hermite_functions(xx, 30)
for n in [0, 5, 15, 28]:
 norm = np.sum(psi[n]**2) * dx
 print(f"||psi_{n}||^2 = {norm:.8f}")


||psi_0||^2 = 1.00000000
||psi_5||^2 = 1.00000000
||psi_15||^2 = 1.00000000
||psi_28||^2 = 1.00000000


In [13]:

# Build M_zeros[m,n] = sum_gamma phi_m(gamma) phi_n(gamma)
# phi_n(t) = sigma^{-1/2} psi_n((t-T0)/sigma).
# So M_zeros = (1/sigma) sum_gamma psi_m(x_g) psi_n(x_g), where x_g = (gamma-T0)/sigma.
# = (1/sigma) Psi @ Psi.T where Psi[n, g] = psi_n(x_g).

def M_zeros_matrix(zeros_list, T0, sigma, J):
 """zeros_list: list of mpmath gammas (real). Compute via float64 (sufficient for traces ~ O(J)).
 Return JxJ float64 matrix."""
 x = np.array([(float(g) - T0)/sigma for g in zeros_list], dtype=np.float64)
 Psi = hermite_functions(x, J) # (J, n_zeros)
 return (Psi @ Psi.T) / sigma

# Test: at T0=85.7, sigma=2, J=10 -- engine reference behavior says |λ_min|/tr ≈ 1e-8..1e-10 (numerical floor) for ζ.
# We need to build M_arith to compute Q. Let's first set up arith and validate trace.
# tr(M_zeros) = (1/sigma) sum_gamma sum_n psi_n((gamma-T0)/sigma)^2.
# Note tr(M_zeros) at T0=85.7, sigma=2, J=10 -- compute it:
T0_t, sigma_t, J_t = 85.7, 2.0, 10
Mz_t = M_zeros_matrix(zeta_zeros, T0_t, sigma_t, J_t)
print(f"tr(M_zeros) at T0={T0_t}, σ={sigma_t}, J={J_t}: {np.trace(Mz_t):.6f}")
print(f"M_zeros shape: {Mz_t.shape}")


tr(M_zeros) at T0=85.7, σ=2.0, J=10: 3.898295
M_zeros shape: (10, 10)


In [14]:

# Now M_arith. Components:
# (A) prime-power sum: M_arith_prime[m,n] = -2 sum_{p,k>=1, p^k<=X} (log p / p^{k/2}) Re[a_L(p^k) F_{mn}(k log p)]
# where F_{mn}(u) = e^{-iu T0} * F0_{mn}(u sigma)
# (B) archimedean term: archim[m,n] = (1/(2pi)) int_{-inf}^{inf} phi_m(r) phi_n(r) * W_arch(r) dr
# W_arch is real-valued; for self-dual L, the integrand is r-symmetric integrand near 0;
# We integrate over the real line, splitting at r=0 for stability (per spec note).
# (C) polar term for ζ: phi_m(i/2) phi_n(i/2) + phi_m(-i/2) phi_n(-i/2) (analytic continuation of phi)
# Need analytic continuation of Hermite-Gauss. phi_n(t) is entire, so phi_n(±i/2) is well-defined.
# (D) g(0) log(q/pi) constant: for L(chi), q=5; for zeta, related to log pi.
# 
# Let me first build the prime sum since it dominates. Need primes up to X=5e7.
# Number of primes up to 5e7 ≈ 5e7/ln(5e7) ≈ 2.8M. Storing as numpy int64 array ~22MB. Fine.

X = int(5e7)
prime_cache = '/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/primes_5e7.npy'
if os.path.exists(prime_cache):
 primes = np.load(prime_cache)
 print(f"Loaded {len(primes)} primes from cache")
else:
 t0 = time.time()
 # Sieve of Eratosthenes
 sieve = np.ones(X+1, dtype=bool)
 sieve[:2] = False
 for i in range(2, int(X**0.5)+1):
 if sieve[i]:
 sieve[i*i::i] = False
 primes = np.nonzero(sieve)[0].astype(np.int64)
 np.save(prime_cache, primes)
 print(f"Generated {len(primes)} primes up to {X} in {time.time()-t0:.1f}s")
print(f"First 10 primes: {primes[:10]}")
print(f"Last prime: {primes[-1]}")


Generated 3001134 primes up to 50000000 in 0.4s
First 10 primes: [ 2 3 5 7 11 13 17 19 23 29]
Last prime: 49999991


In [15]:

# Now design M_arith.
# 
# Conventions: Weil's explicit formula (Iwaniec-Kowalski 5.3 / Theorem 5.12)
# For an L-function L with completed Lambda(s) = q^{s/2} * gamma_inf(s) * L(s), satisfying functional equation Lambda(s) = eps Lambda*(1-s).
# Zeros rho = 1/2 + i gamma (all on critical line under GRH).
# 
# For an even Schwartz test function h on R, with g(u) = (1/(2pi)) int h(r) e^{-iur} dr:
# sum_rho h(gamma) = [contributions from poles] + (1/(2pi)) int h(r) Phi(r) dr 
# - sum_{n>=1} Lambda_L(n) / sqrt(n) (g(log n) + chi_complement * g(-log n))
# 
# For ζ: Lambda_zeta(n) = Lambda(n) (von Mangoldt). Poles at s=0,1 give h(i/2)+h(-i/2). Phi(r) = -log(pi) + (1/2)(psi(1/4+ir/2) + psi(1/4-ir/2)).
# For L(chi): Lambda_chi(n) = chi(n) Lambda(n). No poles. Phi(r) = log(q/pi) + (1/2)(psi((1+a)/2 + ir)/... wait
# gamma_inf(s) = pi^{-(s+a)/2} Gamma((s+a)/2), a=0 or 1. So log derivative: -1/2 log pi + (1/2) psi((s+a)/2).
# On critical line s=1/2+ir: (-1/2 log pi + (1/2) psi((1/2+a)/2 + ir/2)) and complex conjugate from the dual side.
# Conductor contribution: (1/2) log q for each side.
# 
# For NON-self-dual L: we sum h(gamma) over zeros of L AND zeros of L*=L(s,chibar). The zeros of L*(s) at 1/2+i gamma* correspond to zeros of L at 1/2-i gamma* (or 1/2+i(-gamma) where -gamma is in conj). Convention here: list of zeros of L itself = pos + neg lists.
# 
# Let's adopt the SYMMETRIC version. Define the explicit formula identity to be used as basis for:
# M_zeros[m,n] := sum_{gamma zero of L} phi_m(gamma) phi_n(gamma)
# where we use bilinear form h(r) = phi_m(r) phi_n(r) (not squared, but bilinear).
# 
# But wait: h(r) = phi_m(r)phi_n(r) is not even unless T0=0. The standard Weil formula assumes EVEN test function.
# 
# We need to adapt: the explicit formula for an L-function works for general (not just even) test functions h, provided we sum over ALL zeros (including complex conjugates). For non-self-dual L, the zeros are not symmetric about 0 in imaginary part.
# 
# Form (general): for h analytic in strip and decaying, 
# sum_{rho zero of L} h(gamma_rho) = [poles of Lambda] + (1/2pi) int h(r) [(d/dr) arg gamma_inf(1/2+ir) terms ] 
# - sum_n Lambda_chi(n)/sqrt(n) g(log n) - sum_n Lambda_chibar(n)/sqrt(n) g(-log n)
# 
# More precisely: the functional equation gives Lambda(s) = eps Lambda*(1-s), where Lambda*(s) corresponds to L(s,chibar).
# Taking log derivative and applying to test function gives the symmetric formula.
# 
# Practical approach: use the formula in the form (from Bombieri-Hejhal, Stopple, etc.):
# 
# For ANY test function h smooth+decay, define g(u) = (1/(2pi)) int h(r) e^{-iur} dr.
# Then:
# sum_{rho zero L} h(gamma_rho) + [add zeros of L* if non-self-dual, with negated argument]
# = poles + (1/2pi) int h(r) ARG_term(r) dr - sum_n (Lambda(n)/sqrt(n)) [chi(n) g(log n) + chibar(n) g(-log n)]
#
# Then for self-dual L (zeta, chi-real): sum over zeros of L = sum of pairs ±gamma; for non-self-dual, we sum over both L and L*.
# 
# I'll adopt the following clean formulation:
#
# Theorem (Weil explicit formula for L(s, chi) primitive Dirichlet char, no poles):
# sum_{rho: Lambda(rho,chi)=0} h(gamma_rho) 
# = (1/(2pi)) int_R h(r) * [log(q/pi) + Re psi((1/2+a)/2 + ir/2)] dr 
# - sum_{n>=1} Lambda(n)/sqrt(n) [chi(n) g(log n) + chibar(n) g(-log n)]
# Wait that's only counting zeros of Lambda(s,chi), not the conjugate. Let me re-check.
#
# Actually, here's the cleanest derivation (cf. Davenport, Multiplicative Number Theory, Ch. 17):
# For an L-function with functional equation Lambda(s) = eps W Lambda*(1-s), and h(r) of appropriate properties:
# sum_{Lambda(rho)=0} h(gamma_rho) = (poles of Lambda) + (Mellin inversion terms involving gamma_inf and conductor)
# - sum_n Lambda(n)chi(n)/sqrt(n) g(log n) + (terms from the functional eq side)
#
# The full formula (Mestre's identity) is, for an L-function with functional eq Lambda(s,chi) ~ Lambda(1-s, chibar):
# sum_{rho zeros of Lambda(s,chi)} h(gamma_rho) =
# (1/2pi) int_R h(r) [log(q/pi) + (1/2)(psi((1/2+ir+a)/2) + psi((1/2-ir+a)/2))] dr
# - sum_n Lambda(n)/sqrt(n) [chi(n) g(log n) + chibar(n) g(-log n)]
# Hmm but the psi terms with ir and -ir come from the LHS gamma_inf factor only; the conjugate is for chibar which has same gamma factor (since a depends on chi(-1) and chibar(-1)=conj(chi(-1)) — both same for real-valued chi(-1) like our case chi(-1)=-1 either way).
# 
# Let me try once more, carefully: Define Lambda(s,chi) = (q/pi)^{(s+a)/2} Gamma((s+a)/2) L(s,chi).
# Log derivative: Lambda'/Lambda(s) = (1/2) log(q/pi) + (1/2) psi((s+a)/2) + L'/L(s).
# Functional equation: Lambda(s, chi) = eps Lambda(1-s, chibar). So Lambda'/Lambda(s,chi) = - Lambda'/Lambda(1-s,chibar).
# Add them and divide by 2:
# (1/2)[Lambda'/Lambda(s,chi) + Lambda'/Lambda(1-s,chibar)] = (1/2)[ (1/2) log(q/pi) + (1/2) psi((s+a)/2) + L'/L(s,chi)
# + (1/2) log(q/pi) + (1/2) psi((1-s+a)/2) + L'/L(1-s,chibar)]
# = (1/2) log(q/pi) + (1/4)(psi((s+a)/2) + psi((1-s+a)/2)) + (1/2)(L'/L(s,chi) + L'/L(1-s,chibar))
# But by functional eq this also equals zero (since the LHS is antisymmetric under s->1-s + chi->chibar).
# So no useful immediate identity; need to apply Cauchy.
#
# By residue theorem on a rectangular contour with zeros and poles inside:
# sum_{rho} h(gamma_rho) where gamma_rho = (rho-1/2)/i
# Actually I think the cleanest formulation is this:
#
# Let phi(s) be a test function analytic in some strip, decaying. Define h(r) = phi(1/2+ir). Then:
# sum_{rho zero of Lambda(s,chi)} phi(rho) = (sum over poles)
# + (1/2pi i) int_(c) phi(s) Lambda'/Lambda(s,chi) ds on the right
# + similar on the left (after func eq).
# Result:
# sum_rho phi(rho) = phi(1) + phi(0)[for zeta] 
# - (1/(2pi)) int_R h(r) [log(q/pi) + Re psi((1/2+a)/2 + ir)] dr ... wait sign issues
# - sum_n Lambda(n) chi(n) / sqrt(n) * Phi(log n) + dual terms
# where Phi is suitable transform.
# 
# Given the difficulty of getting all signs/factors right from scratch, let me adopt a direct verification approach.
# I'll write the formula with parameters and DETERMINE the correct signs/factors by requiring the trace identity to hold.
# This is justifiable since the trace identity is a known validation gate. I just need ONE valid configuration to fix all signs.
print("Going to determine signs empirically via trace identity validation.")


Going to determine signs empirically via trace identity validation.


In [16]:

# Let me state the formula I'll use (Weil's explicit formula, standard form):
# 
# For any L-function in the Selberg class (here ζ or L(χ)):
# Let h(r) be test function on R (NOT required to be even).
# Then
# Sum_{rho: Lambda(rho)=0, rho=1/2+i gamma} h(gamma) =
# [polar terms: h(-i/2) + h(i/2) for ζ; 0 for L(χ)]
# + (1/(2 pi)) int_R h(r) Phi_arch(r) dr
# - Sum_{n>=2} Lambda(n)/sqrt(n) * [a_L(n) * g_+(log n) + a_L_bar(n) * g_-(log n)]
# where g_+(u) = (1/(2pi)) int h(r) e^{-iur} dr, g_-(u) = (1/(2pi)) int h(r) e^{+iur} dr = g_+(-u).
# For self-dual (zeta, real chi), a_L_bar = a_L (real), so the bracket = a_L(n) (g_+ + g_-)(log n).
# For our zeta: a_L(n) = 1 for n=p^k (so Lambda_zeta = von Mangoldt).
# For L(chi): a_L(p^k) = chi(p^k), a_Lbar(p^k) = chibar(p^k).
#
# Phi_arch(r) = log(C/pi) + Re Psi(s_archim) for self-dual; for ζ specifically Phi_arch(r) = -log(pi) + Re Psi(1/4 + i r/2).
# For L(chi mod 5), with a=1 (odd char): gamma factor is pi^{-(s+1)/2} Gamma((s+1)/2). On critical line s=1/2+ir:
# gamma_inf'/gamma_inf = -1/2 log pi + (1/2) Psi((s+1)/2) = -1/2 log pi + (1/2) Psi(3/4 + ir/2).
# Total archimedean for log(Lambda) on critical line: 1/2 log(q/pi) + 1/2 Psi((1/2+a)/2 + ir/2) on the chi side.
# Combined with functional eq side: total Phi_arch(r) = log(q/pi) + Re Psi((1+a)/2/... 
# Let me just write:
# Phi_arch(r) = log(q/pi) + 2 * Re[ (1/2) Psi((1/2 + a)/2 + i r/2) ]
# = log(q/pi) + Re Psi((1+2a)/4 + ir/2)
# wait... (1/2 + a)/2 = 1/4 + a/2. With a=0: 1/4 (matches zeta). With a=1: 3/4.
# OK so:
# Phi_arch(r) = log(q/pi) + Re Psi(1/4 + a/2 + ir/2)
# (factor of 2 from sum of chi and chibar sides... but they have same real part since psi has nice properties)
# 
# Hmm wait I'm being sloppy with factor of 2. Let me reconsider.
# The contour integral picks up both sides. Each gamma_inf log derivative contributes (1/2) psi((s+a)/2).
# On critical line s=1/2+ir: (1/2) psi(1/4 + a/2 + ir/2).
# Plus from func eq: (1/2) psi(1/4 + a/2 - ir/2) (for chibar, same a).
# Sum = (1/2)(psi(1/4 + a/2 + ir/2) + psi(1/4 + a/2 - ir/2)) = Re psi(1/4 + a/2 + ir/2) (since arg is z and zbar gives 2 Re).
# Plus -log(pi) from both (each contributes -1/2 log pi).
# Plus conductor log q (from both, +1/2 log q each, total log q for chi mod 5).
# 
# So Phi_arch(r) = log(q) - log(pi) + Re psi(1/4 + a/2 + ir/2)
# = log(q/pi) + Re psi(1/4 + a/2 + ir/2)
# For ζ: q=1, a=0 → Phi_arch(r) = -log(pi) + Re psi(1/4 + ir/2). ✓
# For L(chi mod 5, a=1): Phi_arch(r) = log(5/pi) + Re psi(3/4 + ir/2). ✓
# 
# Now the prime sum side:
# - sum_n Lambda(n)/sqrt(n) [a_L(n) g_+(log n) + a_Lbar(n) g_-(log n)]
# where Lambda(n) is von Mangoldt for ζ; for L(chi), the relevant coefficient is the Dirichlet series for -L'/L(s,chi) = sum chi(n) Lambda(n) n^{-s}.
# So the formula uses chi(n) Lambda(n) for the chi side, and chibar(n) Lambda(n) for the conjugate side.
# 
# For ζ: a_L(n) = 1 (chi=trivial), a_Lbar = 1, so the bracket = g_+(log n) + g_-(log n).
# For L(chi mod 5): bracket = chi(n) g_+(log n) + chibar(n) g_-(log n).
# 
# Now in our bilinear setup, h_{m,n}(r) = phi_m(r) phi_n(r).
# g_+(u) = (1/(2pi)) int phi_m(r) phi_n(r) e^{-iur} dr = (1/(2pi)) F_{mn}(u) = (1/(2pi)) e^{-iuT0} F0_{mn}(u sigma).
# 
# So:
# M_arith_prime[m,n] = - sum_{p^k <= X} (log p / p^{k/2}) * [a_L(p^k) (1/(2pi)) F_{mn}(k log p) + a_Lbar(p^k) (1/(2pi)) F_{mn}(-k log p)]
# Note F_{mn}(-u) = conj(F_{mn}(u)) for REAL h_{mn} (since phi's are real). So:
# chi(p^k) F(u) + chibar(p^k) F(-u) = chi(p^k) F(u) + conj(chi(p^k)) conj(F(u)) = 2 Re[chi(p^k) F(u)]
# (this works for both ζ and L(chi)).
# 
# So:
# M_arith_prime[m,n] = -(1/pi) sum_{p^k <= X} (log p / p^{k/2}) Re[chi(p^k) e^{-i k T0 log p} F0_{mn}(k sigma log p)]
# 
# Now the polar term for ζ (poles of Lambda_zeta at s=0,1):
# h(i/2)+h(-i/2) for the sum identity means: contribution to LHS is h(±i/2) ON THE RHS of explicit formula (with the SIGN inherited from derivation; conventionally written as additive on RHS):
# sum_rho h(gamma) = h(i/2) + h(-i/2) + arch - primes
# Therefore the "polar terms" go to M_arith as +phi_m(i/2)phi_n(i/2) + phi_m(-i/2)phi_n(-i/2). 
# Wait: we have M_zeros = M_arith. Or M_zeros - M_arith = 0. The identity is:
# sum_gamma h(gamma) = polar + arch - primes -- this is RHS of explicit formula.
# So define M_arith = polar + arch + prime_part where prime_part = -sum...
# Then sum_gamma h(gamma) - M_arith = 0 if GRH holds.
# 
# Hermite-Gauss analytic continuation: phi_n(t) = sigma^{-1/2} psi_n((t-T0)/sigma), analytic, evaluated at t=±i/2:
# x = (±i/2 - T0)/sigma. Then psi_n(x) = (2^n n! sqrt(pi))^{-1/2} exp(-x^2/2) H_n(x).
# Use the recurrence (or hermite_functions extended to complex). 
# In our case T0 ~ 30..120, sigma=1, so x = (-T0 ± i/2)/1, x ≈ ±i/2 - T0. |x|~T0=O(100). 
# exp(-x^2/2): x^2 = (T0 - i/2)^2 ≈ T0^2 - i T0 - 1/4. exp(-x^2/2) = exp(-T0^2/2) * exp(iT0/2 + 1/8). 
# Hugely small magnitude exp(-T0^2/2) ~ exp(-5000) for T0=100 — completely negligible.
# So polar term contributes negligibly for our T0's at sigma=1.
# Let's confirm numerically and then we can set polar=0 (or include exact value, which will be tiny).
# 
# Archimedean term:
# arch[m,n] = (1/(2pi)) int_R phi_m(r) phi_n(r) Phi_arch(r) dr
# Phi_arch(r) = log(q/pi) + Re psi(1/4 + a/2 + ir/2)
# 
# At T0~30..120, integrand is sharply peaked near T0 (Hermite-Gauss support ~T0±~7). 
# Substitute r = T0 + sigma x: 
# phi_m(r) phi_n(r) = (1/sigma) psi_m(x) psi_n(x), dr = sigma dx
# arch[m,n] = (1/(2pi)) int (1/sigma) psi_m(x) psi_n(x) Phi_arch(T0+sigma x) sigma dx
# = (1/(2pi)) int psi_m(x) psi_n(x) Phi_arch(T0+sigma x) dx
# We can numerically integrate over x in [-15, 15] (range where psi's have support) using a Gauss-Hermite-like or simply Simpson on dense grid.

# Now let me code it up. Let me start with ζ at T0=85.7, σ=2, J=10 to validate against reference behavior (|λ_min|/tr ≈ 1e-8).

print("Implementing M_arith for ζ at reference point (T0=85.7, σ=2, J=10) for validation.")


Implementing M_arith for ζ at reference point (T0=85.7, σ=2, J=10) for validation.


In [17]:

# === Prime sum computation (vectorized in p, looped in (m,n)) ===
# For each (m,n), we need:
# prime_part[m,n] = -(1/pi) sum_{p^k} (log p / p^{k/2}) Re[chi(p^k) e^{-ik T0 log p} F0_{mn}(k sigma log p)]
# 
# For zeta: chi(p^k)=1.
# For L(chi mod 5): chi(p^k) = chi(p)^k. chi(0)=0, chi(5)=0.
# 
# Compute v_k = k sigma log p for each (p, k).
# F0_{mn}(v): we computed analytic form. For large J, max m,n=J-1=27.
# For each k, the contributions decay rapidly in p^{k/2}. So k=1 dominates; k=2 contributes; k>=3 negligible.

# How many (p,k) terms with p^k <= X=5e7?
# k=1: ~3e6 primes
# k=2: primes up to sqrt(5e7) ≈ 7071, so ~900 primes 
# k=3: primes up to (5e7)^(1/3) ≈ 368, so ~73 primes
# k=4: primes up to (5e7)^(1/4) ≈ 84, so ~23 primes
# k=5: ~16, k=6: ~6, k=7: ~4, k=8: ~2
# Higher k: a handful
# So dominated by k=1 with 3M primes. With J=28: 28*28=784 (m,n) pairs (symmetric so 28*29/2=406 distinct).
# Each pair requires evaluating F0_{mn} at 3M points => 3M * 406 ~ 1.2e9 evaluations. With numpy vectorization should be OK but might be slow.
# 
# Per (m,n) pair, F0_{mn}(v) = e^{-v^2/4} * (-iv/sqrt(2))^{N-M} * sqrt(M!/N!) * L_M^{N-M}(v^2/2)
# Hmm, factor with i^{N-M}: this is a complex number.
# 
# Better approach: compute, for each prime p (vectorized array), the full vector of contributions.
# Group all primes/prime powers first, then loop over (m,n).
# 
# Memory considerations: 3M primes -> array of log p, p^{-k/2}, k sigma log p (v_k array), and chi(p^k) for k.
# All as float64/complex128. ~24MB each. Fine.
# 
# Strategy:
# 1. Compute v_grid (array of all k sigma log p values), w_grid (-(log p)/p^{k/2}/pi as weight), phase_grid (chi(p^k) e^{-ik T0 log p}).
# 2. Precompute exp(-v^2/4), v^2/2, e^{-iv T0/sigma * sigma} = e^{-iuT0} where u = k log p.
# Wait: F_{mn}(u) = e^{-iuT0} F0_{mn}(u sigma). So we need:
# overall factor per (p,k): -(log p / p^{k/2}) / pi * Re[chi(p^k) e^{-i k T0 log p} F0_{mn}(k sigma log p)]
# 3. Vectorize over (p,k) for each (m,n).
# 
# Let's precompute everything once for given (T0, sigma, X, L):
# v_arr = k * sigma * log(p) (array over all valid (p,k))
# w_arr = -(log p / p^{k/2}) / pi
# z_arr = chi(p^k) * exp(-i * k * T0 * log p) (complex)
# Then for each (m,n): F0_arr = F0_mn(m, n, v_arr) -- vectorized evaluation
# prime_part[m,n] = sum_p Re(w_arr * z_arr * F0_arr) 
# But careful: w_arr is real, so = sum_p w_arr * Re(z_arr * F0_arr).
# 
# Cost: per (m,n) pair, F0 eval at ~3M points: ~3M complex multiplications + Laguerre eval.
# Laguerre L_M^{alpha}(x) eval at 3M points: ~3M operations per evaluation, using scipy.eval_genlaguerre is fast.
# So per (m,n): ~few hundred ms. For J=28 (406 pairs): ~minutes total per (T0, sigma) config.
# Total grid: 5 T0 × 7 J × 2 L-functions = 70 configs. But we can SHARE the prime arrays across T0 (different z_arr).
# And we can share F0_arr evaluations across all configs (only J and sigma matter). 
# Better strategy:
# For each (sigma, J): F0 matrix evaluated at v_arr -- depends only on sigma (through v_arr = k sigma log p) and (m,n).
# For each T0: phase z_arr = chi(p^k) exp(-i k T0 log p).
# Combine per config.
# 
# At sigma=1 only, v_arr fixed. So we can compute F0_mn(v_arr) ONCE per (m,n), store as a complex array.
# Memory: 406 (m,n) pairs * 3e6 entries * 16 bytes = 19 GB. Too much.
# 
# Alternative: stream over (m,n) and accumulate per-T0 sums. 
# For each (m,n), evaluate F0_arr (single array of ~3M complex), then for each T0, dot product:
# prime_part_T0[m,n] = sum_p w_arr * Re(z_arr_T0 * F0_arr)
# This works: 406 (m,n) × 5 T0 = 2030 sums of 3M elements = 6e9 ops. ~60s with numpy. Fine.
# But evaluating F0 itself dominates if Laguerre eval is slow per (m,n). Let's profile.
# 
# Actually we have J=28 and we need (m,n) up to (27,27). The Laguerre eval per (m,n) is the bottleneck.

# Let me first test the implementation on the reference point T0=85.7, σ=2, J=10 with smaller X to verify correctness.

import scipy.special as sps

def build_prime_kpow_arrays(primes_arr, X):
 """Build flat arrays of (p, k, log_p, p_to_minus_k_half, k_log_p) for all p^k <= X."""
 p_list = []
 k_list = []
 for k in range(1, 32):
 # p^k <= X => p <= X^(1/k)
 lim = X**(1.0/k)
 if k == 1:
 mask = primes_arr <= X
 else:
 mask = primes_arr <= lim + 1
 p_k = primes_arr[mask]
 # Verify p^k <= X exactly (for safety on borderline)
 if k > 1:
 p_k = p_k[p_k.astype(np.int64)**k <= X]
 if len(p_k) == 0:
 break
 p_list.append(p_k)
 k_list.append(np.full(len(p_k), k, dtype=np.int32))
 p_arr = np.concatenate(p_list).astype(np.float64)
 k_arr = np.concatenate(k_list)
 log_p = np.log(p_arr)
 weight = -log_p / np.sqrt(p_arr**k_arr) / np.pi # but p^k could overflow... use p^{k/2} 
 p_to_kh = p_arr**(k_arr/2.0)
 weight = -log_p / p_to_kh / np.pi
 return p_arr, k_arr, log_p, weight

t0 = time.time()
p_arr, k_arr, log_p_arr, weight_arr = build_prime_kpow_arrays(primes, X)
print(f"Built prime-power arrays in {time.time()-t0:.1f}s. Total terms: {len(p_arr)}")
print(f"k counts: {np.bincount(k_arr)[1:]}") 


Built prime-power arrays in 0.3s. Total terms: 3002195
k counts: [3001134 908 73 23 11 8 5 4 4
 3 3 2 2 2 2 2 1 1
 1 1 1 1 1 1 1]


In [18]:

# Define chi function for L(chi mod 5)
# chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1, chi(5)=0
chi_table = {0: 0, 1: 1, 2: 1j, 3: -1j, 4: -1}

def chi_eval(n_arr):
 """Eval chi(n) for array of integers."""
 out = np.zeros(len(n_arr), dtype=np.complex128)
 rem = n_arr % 5
 out[rem == 1] = 1
 out[rem == 2] = 1j
 out[rem == 3] = -1j
 out[rem == 4] = -1
 return out

# For p^k: chi(p^k) = chi(p)^k
def chi_p_power(p_arr, k_arr):
 chip = chi_eval(p_arr.astype(np.int64))
 return chip**k_arr

# For zeta: coef = 1 always
def coef_zeta(p_arr, k_arr):
 return np.ones(len(p_arr), dtype=np.complex128)

# Test
test_p = np.array([2,3,5,7,11,13])
test_k = np.array([1,1,1,2,1,1])
print("chi(p^k) test:", chi_p_power(test_p, test_k))
# chi(2)=i, chi(3)=-i, chi(5)=0, chi(7^2)=chi(7)^2=chi(2)^2=-1 (since 7 mod 5 =2), chi(11)=chi(1)=1, chi(13)=chi(3)=-i


chi(p^k) test: [ 0.+1.j -0.-1.j 0.+0.j -1.+0.j 1.+0.j -0.-1.j]


In [19]:

# Good. Now F0_{mn}(v_arr) vectorized.
# F0_{mn}(v) = e^{-v^2/4} * (-iv/sqrt(2))^{N-M} * sqrt(M!/N!) * L_M^{N-M}(v^2/2)
# Use scipy.special.eval_genlaguerre for L_M^alpha(x) -- supports vectorized x.

def F0_mn_vec(m, n, v_arr):
 """Vectorized F0_{mn} over v_arr (real positive). Returns complex array."""
 M = min(m,n); N = max(m,n); d = N - M
 if d == 0:
 prefac_sqrt = 1.0
 else:
 log_fac = lgamma(N+1) - lgamma(M+1) # log(N!/M!)
 prefac_sqrt = np.exp(-0.5 * log_fac)
 v2 = v_arr * v_arr
 lag = sps.eval_genlaguerre(M, d, v2/2.0)
 base = np.exp(-v2/4.0) * prefac_sqrt * lag
 if d == 0:
 return base.astype(np.complex128)
 # (-iv/sqrt(2))^d = (v/sqrt(2))^d * (-i)^d
 poly = (v_arr / np.sqrt(2.0))**d
 minus_i_d = (-1j)**d
 return base * poly * minus_i_d

# Quick check
v_test = np.array([0.5, 1.0, 2.0])
for (m,n) in [(0,0),(1,1),(0,2),(2,5),(3,7)]:
 a = F0_mn_vec(m,n,v_test)
 print(f"m={m}, n={n}: F0(v_test)={a}")


m=0, n=0: F0(v_test)=[0.93941306+0.j 0.77880078+0.j 0.36787944+0.j]
m=1, n=1: F0(v_test)=[ 0.82198643+0.j 0.38940039+0.j -0.36787944+0.j]
m=0, n=2: F0(v_test)=[-0.08303317+0.j -0.27534766+0.j -0.5202601 +0.j]
m=2, n=5: F0(v_test)=[0.+0.0502897j 0.+0.27104763j 0.+0.26866116j]
m=3, n=7: F0(v_test)=[0.01642387+0.j 0.17032409+0.j 0.28770906+0.j]


In [20]:

# Build the prime-sum part of M_arith.
# prime_part[m,n] = sum over (p,k) of weight * Re[ coef * exp(-i k T0 log p) * F0_mn(k sigma log p) ]
# We need precomputed: v_arr = k_arr * sigma * log_p_arr; klogp_arr = k_arr * log_p_arr.
# 
# For efficiency: precompute u_arr = k_arr * log_p_arr (constant for given prime data, indep of T0, sigma).
# v_arr = sigma * u_arr.
# z_T0 = exp(-i * u_arr * T0). Per T0.
# coef = coef_func(p, k) (per L-function).

def compute_M_arith_prime(p_arr, k_arr, log_p_arr, weight_arr, coef_arr, T0, sigma, J):
 """Compute prime-sum contribution to M_arith. Vectorized over primes; loops over (m,n)."""
 u_arr = k_arr * log_p_arr # length n_terms, real
 v_arr = sigma * u_arr # arg of F0
 phase = np.exp(-1j * u_arr * T0) # complex, length n_terms
 full_coef = coef_arr * phase # complex
 M = np.zeros((J, J), dtype=np.float64)
 for m in range(J):
 for n in range(m, J):
 f0 = F0_mn_vec(m, n, v_arr) # complex array
 val = np.sum(weight_arr * np.real(full_coef * f0))
 M[m, n] = val
 if m != n:
 M[n, m] = val # symmetric since Re[F_{mn}] = Re[F_{nm}]; phi_m*phi_n is symmetric in m,n; F0_{mn}=F0_{nm}
 return M

# Test on ζ at T0=85.7, σ=2, J=10
t0 = time.time()
coef_arr_zeta = coef_zeta(p_arr, k_arr)
M_arith_prime_test = compute_M_arith_prime(p_arr, k_arr, log_p_arr, weight_arr, coef_arr_zeta, 85.7, 2.0, 10)
print(f"Computed M_arith_prime (zeta, ref point) in {time.time()-t0:.1f}s")
print(f"trace(M_arith_prime) = {np.trace(M_arith_prime_test):.6f}")
print(f"trace(M_zeros) for ζ at ref = 3.898295")


Computed M_arith_prime (zeta, ref point) in 7.9s
trace(M_arith_prime) = -0.258203
trace(M_zeros) for ζ at ref = 3.898295


In [21]:

# Now archimedean and polar:
# Polar (zeta only): h(i/2)+h(-i/2) where h(r)=phi_m(r)phi_n(r).
# h(±i/2) = phi_m(±i/2)phi_n(±i/2). For T0=85.7, σ=2: phi_n(±i/2) involves exp(-((±i/2-T0)/σ)^2/2).
# x = (±i/2 - 85.7)/2 ≈ -42.85 ± i/4. x^2 ≈ 1836 mp i, exp(-x^2/2) ~ exp(-918) — beyond underflow.
# So polar is utterly negligible for our parameters.
#
# Archimedean: arch[m,n] = (1/(2pi)) int_R phi_m(r) phi_n(r) Phi_arch(r) dr
# Phi_arch(r) for ζ: -log(pi) + Re psi(1/4 + ir/2)
# Per substitution: arch[m,n] = (1/(2pi)) int psi_m(x) psi_n(x) Phi_arch(T0 + sigma x) dx
# psi_m(x) psi_n(x) decays as exp(-x^2). Use Gauss-Hermite quadrature for max accuracy!

# Gauss-Hermite quadrature: int e^{-x^2} f(x) dx ≈ sum w_i f(x_i) using Gauss-Hermite weights/nodes.
# Our integrand psi_m psi_n includes exp(-x^2), so:
# psi_m(x) psi_n(x) = norm * H_m(x) H_n(x) * exp(-x^2)
# So int psi_m psi_n g(x) dx = norm * int H_m(x) H_n(x) e^{-x^2} g(x) dx
# = norm * sum_i w_i H_m(xi) H_n(xi) g(xi) (Gauss-Hermite N-point)
# For polynomial degree m+n exact with N >= (m+n+1)/2+1.
# But g(x) = Phi_arch(T0 + sigma x) is smooth, not polynomial. Need adequately many points.
# Use N=2*J for safety.

# Actually it's cleaner to just integrate the smooth thing directly:
# arch_{mn} = (1/(2pi)) ∫ psi_m(x) psi_n(x) Phi_arch(T0+sigma x) dx
# Use scipy quad on the integrand with the psi's computed via recurrence.
# Or: just use a fine numerical grid with Simpson, since integrand is supported in ~[-10,10].

def archim_integrand_value(T0, sigma, a):
 """Return a function r -> Phi_arch(T0 + sigma r), where r is the substituted variable."""
 # Phi_arch(t) = log(q/pi) + Re psi(1/4 + a/2 + i t/2), but log(q/pi) is constant
 # Use scipy's digamma for complex argument
 from scipy.special import digamma
 if a == 0:
 const = -np.log(np.pi)
 else:
 const = np.log(5.0/np.pi)
 def f(x):
 t = T0 + sigma * x
 z = 1.0/4 + a/2.0 + 1j * t / 2.0
 return const + np.real(digamma(z))
 return f

# Gauss-Hermite based: use psi product, no exp(-x^2) factor explicit.
def archim_matrix_gh(T0, sigma, J, a, n_quad=None):
 """Compute archimedean matrix using Gauss-Hermite quadrature."""
 if n_quad is None:
 n_quad = max(2*J + 20, 60)
 nodes, gh_weights = np.polynomial.hermite.hermgauss(n_quad)
 # Gauss-Hermite: sum w_i f(x_i) ≈ int e^{-x^2} f(x) dx
 # Integrand: psi_m(x) psi_n(x) Phi_arch(T0 + sigma x). 
 # psi_m psi_n = exp(-x^2) * N_m N_n * H_m H_n. So we can split exp(-x^2) and use GH directly:
 # int psi_m psi_n g dx = int e^{-x^2} (N_m N_n H_m H_n g) dx ≈ sum w_i N_m N_n H_m(xi) H_n(xi) g(xi)
 # Easier: just evaluate full psi_n(xi) (already includes exp(-x^2/2)):
 # psi_m(xi) psi_n(xi) = exp(-xi^2) * (N_m H_m N_n H_n).
 # int psi_m psi_n g dx ≈ sum w_i [e^{xi^2} psi_m(xi) psi_n(xi)] * g(xi) (since GH includes e^{-x^2})
 Psi = hermite_functions(nodes, J) # (J, n_quad)
 # Per quadrature pt: psi_n(xi) values. Multiply by exp(xi^2) to extract H_n(xi) N_n.
 H_vals = Psi * np.exp(nodes**2) # (J, n_quad) = N_n H_n(xi)
 f = archim_integrand_value(T0, sigma, a)
 g_vals = np.array([f(x) for x in nodes]) # (n_quad,)
 # arch_{mn} = (1/(2pi)) sum_i gh_weights[i] * H_vals[m,i] * H_vals[n,i] * g_vals[i]
 # Note H_vals already includes N_n normalization
 integrand_per_node = gh_weights * g_vals # (n_quad,)
 M = (H_vals * integrand_per_node) @ H_vals.T / (2*np.pi)
 return M

# But wait: psi_m psi_n integrand is smooth, and the spec says odd-parity basis functions require splitting at the center.
# Reason: H_n is even/odd in x, but Phi_arch is not even in x about x=0 (since x=0 maps to r=T0 ≠ 0).
# Gauss-Hermite should work since it integrates polynomials × exp(-x^2) exactly, and the test function H_m H_n is polynomial.
# For non-polynomial g, we need enough quadrature points to resolve smoothness of g. Digamma is smooth.
# But H_m H_n is polynomial of degree m+n <= 2J-2. So GH with n_quad >= J-ish should integrate exactly the polynomial part.
# For the digamma weight: it's analytic and slowly varying (psi(c+ir/2) for r near T0 is essentially log(T0/2)).
# With n_quad = 2J + 20, should be fine.

t0 = time.time()
arch_test = archim_matrix_gh(85.7, 2.0, 10, a=0)
print(f"arch matrix (zeta, ref) computed in {time.time()-t0:.2f}s")
print(f"trace(arch) = {np.trace(arch_test):.6f}")


arch matrix (zeta, ref) computed in 0.01s
trace(arch) = 624407079230204.625000


In [22]:

# That's huge - quadrature error from oscillatory or large values? Let me check the integrand at nodes.
nodes, gh_w = np.polynomial.hermite.hermgauss(60)
print("Nodes range:", nodes.min(), nodes.max())
f = archim_integrand_value(85.7, 2.0, 0)
g_vals = np.array([f(x) for x in nodes])
print("g values range:", g_vals.min(), g_vals.max())
print("g sample:", g_vals[::10])


Nodes range: -10.159109246180087 10.159109246180087
g values range: 2.342356835530934 2.825730178401928
g sample: [2.34235684 2.4659293 2.54684435 2.61629717 2.68146453 2.74812174]


In [23]:

# g values fine (~2.5). But trace is 6e14. That means my H_vals * exp(xi^2) blows up at large xi.
# Issue: H_vals = Psi * exp(xi^2) where Psi includes exp(-xi^2/2). So H_vals = N_n H_n(xi) exp(xi^2/2). 
# Then we want H_vals[m] H_vals[n] = N_m N_n H_m H_n exp(xi^2). And the GH quad rule gives sum w_i (N_m H_m N_n H_n exp(xi^2)) g_i.
# Wait, the GH approximation is for int e^{-x^2} f(x) dx. So if my f(x) = N_m N_n H_m H_n g, then sum w_i N_m N_n H_m H_n g_i ≈ int e^{-x^2} N_m N_n H_m H_n g dx = int psi_m psi_n g dx? Let me recheck.
# psi_m(x) psi_n(x) = N_m H_m(x) exp(-x^2/2) * N_n H_n(x) exp(-x^2/2) = N_m N_n H_m(x) H_n(x) e^{-x^2}.
# So int psi_m psi_n g dx = int e^{-x^2} (N_m N_n H_m H_n g) dx ≈ sum_i w_i (N_m N_n H_m(xi) H_n(xi) g_i).
# In code, I have Psi[n,i] = psi_n(xi) = N_n H_n(xi) e^{-xi^2/2}. So Psi[m,i] * Psi[n,i] = N_m N_n H_m H_n e^{-xi^2}.
# Therefore the integrand for quadrature is just Psi[m,i] Psi[n,i] times nothing? Hmm wait.
# 
# We have two approaches:
# A) Direct: ∫ psi_m psi_n g dx = sum_i (psi_m(xi) psi_n(xi) g_i) Δxi -- needs trapezoidal/Simpson on dense grid
# B) GH approximation: rewrite as ∫ e^{-x^2} (N_m N_n H_m H_n) g dx ≈ sum gh_w_i [N_m N_n H_m(xi) H_n(xi)] g_i
# The thing in [] = psi_m(xi) psi_n(xi) * e^{xi^2}.
# 
# So actually the multiplication by e^{xi^2} is needed precisely because GH absorbs e^{-x^2} into its weight.
# But at xi=10.16, e^{xi^2}=e^{103} = 2e44, and Psi[m,xi]^2 ~ e^{-103}. Their product is 1 (numerically). But computing them separately is unstable.
# 
# Let's use approach A: dense Simpson grid since the integrand decays as exp(-x^2). Use [-15, 15] with 2000 points.

def archim_matrix_direct(T0, sigma, J, a, n_pts=4000, x_lim=15.0):
 x = np.linspace(-x_lim, x_lim, n_pts+1)
 dx = x[1]-x[0]
 Psi = hermite_functions(x, J) # (J, n_pts+1)
 f = archim_integrand_value(T0, sigma, a)
 g_vals = f(x) # vectorized
 integrand_factor = g_vals # multiply
 # arch[m,n] = (1/2pi) ∫ Psi[m] Psi[n] g(x) dx
 # Use trapezoidal/Simpson
 # Build M = (Psi * integrand_factor) @ Psi.T * dx / (2pi)
 weighted = Psi * integrand_factor # broadcast (J, n)
 # Simpson weights
 w = np.ones_like(x)
 w[1:-1:2] = 4
 w[2:-1:2] = 2
 w_arr_simp = w * dx / 3.0
 M = (Psi * w_arr_simp) @ (weighted.T) / (2*np.pi)
 return M

t0 = time.time()
arch_test_direct = archim_matrix_direct(85.7, 2.0, 10, a=0)
print(f"Direct arch matrix in {time.time()-t0:.2f}s")
print(f"trace = {np.trace(arch_test_direct):.10f}")


Direct arch matrix in 0.00s
trace = 4.1564980228


In [24]:

# Now check trace identity for ζ at T0=85.7, σ=2, J=10:
# tr(M_zeros) = 3.898295
# tr(M_arith) = tr(prime) + tr(arch) + tr(polar) = -0.258203 + 4.156498 + ~0
# = 3.898295. Great match!
print("Trace identity check:")
print(f"tr(M_zeros) = {np.trace(Mz_t):.10f}")
print(f"tr(M_arith) = tr(prime) + tr(arch) = {np.trace(M_arith_prime_test):.10f} + {np.trace(arch_test_direct):.10f}")
print(f" = {np.trace(M_arith_prime_test) + np.trace(arch_test_direct):.10f}")
print(f"residual = {np.trace(Mz_t) - np.trace(M_arith_prime_test) - np.trace(arch_test_direct):.2e}")


Trace identity check:
tr(M_zeros) = 3.8982947981
tr(M_arith) = tr(prime) + tr(arch) = -0.2582032248 + 4.1564980228
 = 3.8982947981
residual = -1.61e-13


In [25]:

# Excellent! Trace identity holds to 1.6e-13 at the reference point. Sign convention all correct.
# Let me check lambda_min for ζ at this reference point. Spec: |λ_min|/tr should be ~1e-8 to 1e-10.

Q_test = Mz_t - (M_arith_prime_test + arch_test_direct)
eigvals = np.linalg.eigvalsh(Q_test)
print(f"All eigvals: {eigvals}")
print(f"λ_min = {eigvals[0]:.6e}")
print(f"|λ_min|/tr(M_zeros) = {abs(eigvals[0])/np.trace(Mz_t):.2e}")


All eigvals: [-2.62432946e-14 -2.27834769e-14 -2.02780406e-14 -1.91400037e-14
 -1.61980437e-14 -1.47042710e-14 -1.26673106e-14 -1.23721119e-14
 -9.33894270e-15 -7.06255379e-15]
λ_min = -2.624329e-14
|λ_min|/tr(M_zeros) = 6.73e-15


In [26]:

# Better than the spec's reference 1e-8 to 1e-10 (we got 1e-15). 
# Now test for L(chi mod 5) at the same reference point to validate.

coef_arr_chi = chi_p_power(p_arr, k_arr)
t0 = time.time()
Mprime_chi_test = compute_M_arith_prime(p_arr, k_arr, log_p_arr, weight_arr, coef_arr_chi, 85.7, 2.0, 10)
print(f"Chi prime sum: {time.time()-t0:.1f}s")

arch_chi_test = archim_matrix_direct(85.7, 2.0, 10, a=1)
# For L(chi): no polar term, plus zeros from both pos and neg lists.

Lchi_zeros_all = Lchi_pos + Lchi_neg
Mz_chi_test = M_zeros_matrix(Lchi_zeros_all, 85.7, 2.0, 10)

print(f"tr(M_zeros) = {np.trace(Mz_chi_test):.10f}")
print(f"tr(M_arith) = {np.trace(Mprime_chi_test) + np.trace(arch_chi_test):.10f}")
print(f"residual: {np.trace(Mz_chi_test) - np.trace(Mprime_chi_test) - np.trace(arch_chi_test):.3e}")

Q_chi = Mz_chi_test - (Mprime_chi_test + arch_chi_test)
eigvals_chi = np.linalg.eigvalsh(Q_chi)
print(f"λ_min(L(chi)) = {eigvals_chi[0]:.4e}, |λ_min|/tr = {abs(eigvals_chi[0])/np.trace(Mz_chi_test):.2e}")


Chi prime sum: 7.7s
tr(M_zeros) = 6.6869909944
tr(M_arith) = 6.6869909944
residual: -2.576e-13
λ_min(L(chi)) = -3.6977e-14, |λ_min|/tr = 5.53e-15


In [27]:

# Both controls pass at reference point with ~1e-15 floor. Now let's test at the target regime: σ=1.
# Let's do one quick test at T0=46.13, σ=1, J=12 (the "optimal" point from spec)
T0, sigma, J = 46.13, 1.0, 12

t0 = time.time()
Mp_zeta = compute_M_arith_prime(p_arr, k_arr, log_p_arr, weight_arr, coef_arr_zeta, T0, sigma, J)
print(f"prime sum ({time.time()-t0:.1f}s)")
arch_zeta = archim_matrix_direct(T0, sigma, J, a=0)
Mz_zeta = M_zeros_matrix(zeta_zeros, T0, sigma, J)
Q_zeta = Mz_zeta - (Mp_zeta + arch_zeta)
ev_zeta = np.linalg.eigvalsh(Q_zeta)
tr_res = np.trace(Mz_zeta) - np.trace(Mp_zeta) - np.trace(arch_zeta)
print(f"ζ at T0={T0}, σ={sigma}, J={J}: tr(Mz)={np.trace(Mz_zeta):.6f}, tr_res={tr_res:.3e}, λ_min={ev_zeta[0]:.3e}, |λ|/tr={abs(ev_zeta[0])/np.trace(Mz_zeta):.2e}")


prime sum (11.6s)
ζ at T0=46.13, σ=1.0, J=12: tr(Mz)=3.690677, tr_res=-1.443e-13, λ_min=-3.467e-14, |λ|/tr=9.39e-15


In [28]:

# At σ=1, J=12 with X=5e7, trace residual is 1.4e-13 — excellent. Below the floor mentioned in spec (~10^-10 with X=1e7-1e8).
# Hypothesis H1 supported here at least at this point.
# 
# Now let's run the full grid. With sigma=1 fixed, we can precompute v_arr = u_arr (sigma=1) once,
# and F0_mn(v_arr) once per (m,n), then loop over T0 with different phases.
# 
# Optimization: factor out F0 computation. 
# For each (m,n), F0_arr is computed once (3M points). Cost is dominated by Laguerre evaluation. 
# Then for each T0: prime_part_T0[m,n] = sum( weight * Re(coef_arr * exp(-i u T0) * F0_arr) ).
# This loop over T0 is cheap (~3 sums of 3M floats per T0 per (m,n)).
# 
# So total cost: J*(J+1)/2 F0 evaluations + (J*(J+1)/2) * n_T0 * 2 array ops.
# For J=28: 406 F0 evals × ~0.06s = ~24s. Plus 406 * 5 * 0.05s = 100s for the T0 reductions.

def compute_prime_grid(p_arr, k_arr, log_p_arr, weight_arr, coef_arr, T0_list, sigma, J):
 """Return dict T0 -> M_arith_prime matrix (JxJ) for the given coef (per L-function) at fixed sigma."""
 u_arr = k_arr * log_p_arr
 v_arr = sigma * u_arr
 # For each T0, precompute exp(-i u T0)
 phase_dict = {T0: np.exp(-1j * u_arr * T0) for T0 in T0_list}
 M_dict = {T0: np.zeros((J, J), dtype=np.float64) for T0 in T0_list}
 for m in range(J):
 for n in range(m, J):
 f0 = F0_mn_vec(m, n, v_arr) # complex
 base = coef_arr * f0 # complex; T0-independent
 for T0 in T0_list:
 phase = phase_dict[T0]
 val = np.sum(weight_arr * np.real(base * phase))
 M_dict[T0][m, n] = val
 if m != n:
 M_dict[T0][n, m] = val
 return M_dict

T0_list = [30.0, 46.13, 60.0, 85.7, 120.0]
J_max = 28
sigma = 1.0

print("Computing ζ prime-grid (sigma=1, all T0, J=28)...")
t0 = time.time()
Mprime_zeta_dict = compute_prime_grid(p_arr, k_arr, log_p_arr, weight_arr, coef_arr_zeta, T0_list, sigma, J_max)
print(f"ζ prime grid done in {time.time()-t0:.1f}s")


Computing ζ prime-grid (sigma=1, all T0, J=28)...


ζ prime grid done in 117.5s


In [29]:

# 117s for ζ. Same for chi. Now build full results for all (T0, J) at sigma=1.

# Save M_arith_prime to disk in case
with open('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Mprime_zeta_sigma1.pkl','wb') as f:
 pickle.dump({float(k): v for k,v in Mprime_zeta_dict.items()}, f)
print("Saved zeta prime matrices.")

# For L(chi):
print("Computing L(chi) prime-grid...")
t0 = time.time()
Mprime_chi_dict = compute_prime_grid(p_arr, k_arr, log_p_arr, weight_arr, coef_arr_chi, T0_list, sigma, J_max)
print(f"L(chi) prime grid done in {time.time()-t0:.1f}s")

with open('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Mprime_chi_sigma1.pkl','wb') as f:
 pickle.dump({float(k): v for k,v in Mprime_chi_dict.items()}, f)


Saved zeta prime matrices.
Computing L(chi) prime-grid...


L(chi) prime grid done in 116.7s


In [30]:

# Now compute archimedean matrices and M_zeros, then assemble Q and compute eigenvalues for full grid.

J_values = [4, 8, 12, 16, 20, 24, 28]
sigma = 1.0
Lchi_zeros_all = Lchi_pos + Lchi_neg

results = [] # rows: (function, T0, sigma, J, X, lambda_min, tr_Mz, tr_Marith, tr_res_rel, n_zeros_used, runtime_s)

t_all = time.time()
for func_name, coef_kind, a_arch, zeros_list, Mp_dict in [
 ('zeta', 'zeta', 0, zeta_zeros, Mprime_zeta_dict),
 ('Lchi', 'chi', 1, Lchi_zeros_all, Mprime_chi_dict),
]:
 for T0 in T0_list:
 # Restrict zeros to relevant window for efficiency (Hermite-Gauss decay)
 # zeros within |γ - T0| < 30 sufficient since at J=28, classical TP ≈ sqrt(2*28)=7.5; safety to 20 sigma.
 zlist_relevant = [g for g in zeros_list if abs(float(g) - T0) < 30]
 n_used = len(zlist_relevant)
 # Compute Psi at the relevant zeros
 x = np.array([(float(g) - T0)/sigma for g in zlist_relevant], dtype=np.float64)
 Psi_full = hermite_functions(x, J_max)
 for J in J_values:
 t0 = time.time()
 # M_zeros via subset of Psi
 Mz = (Psi_full[:J] @ Psi_full[:J].T) / sigma
 # M_arith prime: slice of precomputed matrix
 Mp = Mp_dict[T0][:J, :J]
 # Archimedean
 arch = archim_matrix_direct(T0, sigma, J, a=a_arch)
 Q = Mz - (Mp + arch)
 eigvals = np.linalg.eigvalsh(Q)
 lam_min = eigvals[0]
 tr_Mz = np.trace(Mz)
 tr_Marith = np.trace(Mp) + np.trace(arch)
 tr_res = tr_Mz - tr_Marith
 tr_res_rel = abs(tr_res) / abs(tr_Mz) if tr_Mz != 0 else float('inf')
 runtime = time.time() - t0
 results.append({
 'function': func_name, 'T0': T0, 'sigma': sigma, 'J': J, 'X': X,
 'lambda_min': lam_min, 'trace_M_zeros': tr_Mz, 'trace_M_arith': tr_Marith,
 'trace_residual': tr_res, 'trace_residual_rel': tr_res_rel,
 'n_zeros_used': n_used, 'runtime_s': runtime,
 })
print(f"Total time: {time.time()-t_all:.1f}s")
print(f"Computed {len(results)} configurations.")


Total time: 0.1s
Computed 70 configurations.


In [31]:

import pandas as pd
df = pd.DataFrame(results)
df['abs_lambda_min'] = np.abs(df['lambda_min'])
df['lambda_over_tr'] = df['abs_lambda_min'] / df['trace_M_zeros']
print(df.to_string(index=False, float_format=lambda x: f"{x:.3e}" if abs(x) < 0.01 or abs(x) > 1e4 else f"{x:.6f}"))


function T0 sigma J X lambda_min trace_M_zeros trace_M_arith trace_residual trace_residual_rel n_zeros_used runtime_s abs_lambda_min lambda_over_tr
 zeta 30.000000 1.000000 4 50000000 -1.334e-14 1.009446 1.009446 -3.841e-14 3.805e-14 13 2.331e-03 1.334e-14 1.322e-14
 zeta 30.000000 1.000000 8 50000000 -1.748e-14 2.238963 2.238963 -7.283e-14 3.253e-14 13 1.591e-03 1.748e-14 7.809e-15
 zeta 30.000000 1.000000 12 50000000 -2.810e-14 2.928659 2.928659 -1.124e-13 3.836e-14 13 1.957e-03 2.810e-14 9.595e-15
 zeta 30.000000 1.000000 16 50000000 -1.403e-13 4.212459 4.212459 2.807e-13 6.663e-14 13 1.848e-03 1.403e-13 3.331e-14
 zeta 30.000000 1.000000 20 50000000 -3.849e-10 4.976926 4.976926 9.768e-10 1.963e-10 13 2.421e-03 3.849e-10 7.734e-11
 zeta 30.000000 1.000000 24 50000000 -3.592e-07 5.669894 5.669894 6.969e-07 1.229e-07 13 2.417e-03 3.592e-07 6.335e-08
 zeta 30.000000 1.000000 28 50000000 -1.152e-04 6.376745 6.376575 1.705e-04 2.674e-05 13 2.153e-03 1.152e-04 1.806e-05
 zeta 46.130000 1.

In [32]:

# Save results to CSV
df.to_csv('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/lambda_min_sigma1.csv', index=False)
print("Saved CSV. Showing summary by function and J:")

# Pivot tables
pv_lam = df.pivot_table(index=['function','T0'], columns='J', values='lambda_min')
pv_res = df.pivot_table(index=['function','T0'], columns='J', values='trace_residual_rel')
print("\n=== λ_min (signed) ===")
print(pv_lam.to_string(float_format=lambda x: f"{x:+.2e}"))
print("\n=== trace_residual_rel ===")
print(pv_res.to_string(float_format=lambda x: f"{x:.2e}"))


Saved CSV. Showing summary by function and J:

=== λ_min (signed) ===
J 4 8 12 16 20 24 28
function T0 
Lchi 30.00 -2.19e-14 -2.64e-14 -3.40e-14 -4.75e-14 -9.46e-13 -8.34e-10 -2.64e-07
 46.13 -2.69e-14 -3.66e-14 -4.28e-14 -5.61e-14 -1.79e-12 -1.63e-09 -5.18e-07
 60.00 -3.06e-14 -3.60e-14 -5.20e-14 -6.44e-14 -4.50e-13 -3.94e-10 -1.31e-07
 85.70 -3.92e-14 -4.26e-14 -5.59e-14 -6.31e-14 -7.40e-12 -5.60e-09 -1.49e-06
 120.00 -4.41e-14 -6.17e-14 -7.90e-14 -9.40e-14 -3.40e-12 -2.40e-09 -5.91e-07
zeta 30.00 -1.33e-14 -1.75e-14 -2.81e-14 -1.40e-13 -3.85e-10 -3.59e-07 -1.15e-04
 46.13 -1.39e-14 -2.72e-14 -3.47e-14 -4.05e-13 -9.67e-10 -7.37e-07 -1.96e-04
 60.00 -1.74e-14 -3.58e-14 -4.19e-14 -4.06e-13 -9.10e-10 -6.73e-07 -1.74e-04
 85.70 -2.37e-14 -4.36e-14 -5.48e-14 -9.66e-14 -1.05e-10 -1.02e-07 -3.40e-05
 120.00 -3.97e-14 -6.01e-14 -8.06e-14 -1.41e-13 -1.25e-10 -1.13e-07 -3.50e-05

=== trace_residual_rel ===
J 4 8 12 16 20 24 28
function T0 
Lchi 30.00 3.66e-14 4.32e-14 4.15e-14 4.02e-14 1.35e-1

In [33]:

# IMPORTANT FINDING: At J<=16, the trace residual is at machine precision (~1e-14) and λ_min ~ -1e-14 (numerical floor).
# Starting at J=20, both λ_min and trace residual become substantially larger:
# ζ at J=20: λ_min ≈ -1e-10, trace_residual_rel ≈ 1e-10
# ζ at J=28: λ_min ≈ -1e-4 to -2e-4, trace_residual_rel ≈ 1e-5
# Lchi at J=20: λ_min ≈ -1e-12, trace_residual_rel ≈ 1e-13
# Lchi at J=28: λ_min ≈ -1e-6, trace_residual_rel ≈ 1e-8
# 
# This is the hallmark of float64 precision failure for high Hermite functions. 
# The Hermite-Gauss recurrence loses precision at high n, and the prime sum involves cancellation of huge oscillatory terms.
# We need higher-precision arithmetic for J >= 20 (or use double-double).
# 
# Actually note: the trace residual matches λ_min closely, suggesting the entire numerical floor is set by precision.
# At J=20-28, |λ_min| ≈ |trace residual| ≈ machine precision × characteristic scale of M_arith_prime.
# 
# Let me check the magnitude of individual entries in M_arith_prime at J=28. The prime sum has 3M terms with values up to ~log(p)/sqrt(p) and oscillating. The sum involves catastrophic cancellation:

# Estimate the absolute sum vs the actual sum for one entry
m, n = 27, 27
u_arr = k_arr * log_p_arr
v_arr = sigma * u_arr
f0 = F0_mn_vec(m, n, v_arr)
T0 = 46.13
phase = np.exp(-1j * u_arr * T0)
base = coef_arr_zeta * f0 * phase
contributions = weight_arr * np.real(base)
print(f"Sum of contributions (m=n=27, ζ, T0=46.13): {np.sum(contributions):.6e}")
print(f"Sum of |contributions|: {np.sum(np.abs(contributions)):.6e}")
print(f"Ratio: {np.sum(np.abs(contributions))/abs(np.sum(contributions)):.2e}")
print(f"Catastrophic cancellation factor: 1 part in {np.sum(np.abs(contributions))/abs(np.sum(contributions)):.0e}")
print(f"Expected precision: 1e-16 × abs-sum / actual = {np.sum(np.abs(contributions)) * 1e-16:.2e} = abs error")


Sum of contributions (m=n=27, ζ, T0=46.13): 1.392143e-01
Sum of |contributions|: 7.892051e+01
Ratio: 5.67e+02
Catastrophic cancellation factor: 1 part in 6e+02
Expected precision: 1e-16 × abs-sum / actual = 7.89e-15 = abs error


In [34]:

# Cancellation is 1 part in 600 — modest. But individual matrix entries M[m,n] for off-diagonal at high m,n can be much larger in absolute sum.
# Let me check the worst-case entry to understand the precision loss.

# Print the full M_arith_prime matrix at T0=46.13 for J=28, scan max abs entries.
Mp = Mprime_zeta_dict[46.13]
print(f"Max |Mp| entry: {np.max(np.abs(Mp)):.3e}")
print(f"Max |Mp[m,n]| for high m,n (>=20):")
print(f" m=n=27: {Mp[27,27]:.6e}")
print(f" Max in m,n in [20,27]: {np.max(np.abs(Mp[20:,20:])):.3e}")

# Issue is likely in the Hermite function recurrence at high orders for arguments far from 0.
# psi_n(x) at large |x| oscillates and grows in dynamic range; the recurrence has error growth.
# For x = (gamma - T0)/sigma where γ-T0 can be up to ~10 and J=28, sqrt(2J) ≈ 7.5; we're near the turning point.

# Let me check M_zeros directly - the zeros are not extremely large in number, so M_zeros should be accurate.
# But the Hermite values at zeros are key.

# Issue: in the prime sum, F0_mn(v) is evaluated at v=u, which can be up to ~sigma * k * log(X) = 1 * 1 * log(5e7) = 17.7.
# For large v, F0_mn has the Laguerre polynomial L_M^{N-M}(v^2/2), which for v=17.7 has argument 156.7.
# Laguerre L_27 at large argument has many oscillations and large dynamic range, leading to error in eval_genlaguerre.
# Let me check:
v_big = 17.7
for J in [4, 12, 20, 27]:
 lag = sps.eval_genlaguerre(J, 0, v_big**2/2)
 print(f"L_{J}^0({v_big**2/2:.1f}) = {lag:.6e}")


Max |Mp| entry: 3.813e-01
Max |Mp[m,n]| for high m,n (>=20):
 m=n=27: 1.392143e-01
 Max in m,n in [20,27]: 2.615e-01
L_4^0(156.6) = 2.259788e+07
L_12^0(156.6) = 1.682499e+17
L_20^0(156.6) = 1.684266e+24
L_27^0(156.6) = -5.119314e+28


In [35]:

# That's the issue! L_27 at x=156 is ~5e28 but multiplied by exp(-v^2/4) = exp(-78) = 1.4e-34 to give a small result.
# The product is dominated by cancellation: 5e28 × 1.4e-34 = 7e-6, but each has 16-digit precision in float64,
# so the absolute error in the product is ~5e28 × 1.4e-34 × 1e-16 = 7e-22, but the relative error from L_27 eval at large argument can be much worse.
# 
# Solution: evaluate F0 in higher precision. Or use a stable algorithm: e.g., write F0 in terms of Hermite functions directly (since psi_m psi_n is product of Hermite-Gauss):
# F0_{mn}(v) = sqrt(2pi) sum_k c_{mn}^k (-i)^k psi_k(v/...)?
# Actually note F0_{mn}(v) = int psi_m(x) psi_n(x) e^{-ivx} dx is the Fourier transform of psi_m * psi_n.
# Expand psi_m psi_n = sum_k a_k psi_k using "Clebsch-Gordan" type coefficients (this expansion has degree up to m+n).
# Then F0_{mn}(v) = sqrt(2pi) sum_k a_k (-i)^k psi_k(v) ... but psi_k is also Hermite-Gauss which is bounded.
# 
# This would be much more stable for large v. Let me derive it.
# 
# In quantum mechanics, the matrix elements of e^{-i v x} between SHO eigenstates are well-known displacement operator matrix elements:
# <m| e^{-ivx} |n> = ... involves Laguerre polynomial, same formula essentially.
# 
# Direct stable approach: just use mpmath for evaluating L_M^alpha at large arguments, when stability is needed.
# Or: precompute F0 in mp arithmetic for high-J entries.
# 
# Alternative: since the issue is only at high J (>=20), and computation is fast, let me use mpmath for F0 evaluation.
# 
# But mpmath is too slow for 3M point evaluations. Need a clever approach.
# 
# Best approach: express F0_{mn}(v) recursively in stable form.
# Recurrence for F0: from raising/lowering ops:
# x psi_n(x) = sqrt(n/2) psi_{n-1}(x) + sqrt((n+1)/2) psi_{n+1}(x)
# So psi_m(x) psi_n(x) ... hmm complex.
# 
# Better: use raising/lowering for F0 itself.
# We have F0_{mn}(v) = int psi_m(x) psi_n(x) e^{-ivx} dx. Let D_v = -i ∂_v. Then x e^{-ivx} = -i ∂_v e^{-ivx} = D_v e^{-ivx}.
# Therefore int psi_m psi_n x e^{-ivx} dx = D_v F0_{mn}(v).
# But also x psi_m = sqrt(m/2) psi_{m-1} + sqrt((m+1)/2) psi_{m+1}, so:
# int (x psi_m) psi_n e^{-ivx} dx = sqrt(m/2) F0_{m-1,n}(v) + sqrt((m+1)/2) F0_{m+1,n}(v)
# = D_v F0_{m,n}(v) = -i F0'_{m,n}(v)
# 
# So sqrt(m/2) F0_{m-1,n} + sqrt((m+1)/2) F0_{m+1,n} = -i F0'_{m,n}
# Not immediate.
# 
# Alternative: use the parabolic cylinder / Mehler kernel representation.
# 
# I think the practical solution is: evaluate F0 in HIGHER PRECISION (extended-double, say) only for the matrix entries at high (m,n). Use multiprecision.

# Let me check whether numpy supports complex128 -> longdouble. Could use complex256 (extended). But that may not be enough either.
# 
# Plan: use mpmath ONLY for evaluating F0 at the (m,n) pairs with high m or high n, for the FULL set of primes.
# This is expensive per call, but for J<=16 we have machine precision floor. For J=20-28 we may need 30-digit precision.
# 
# Cost estimate: mp eval per (p, k) pair: ~1ms at dps=30. 3M × 1ms = 3000s per (m,n). For ~(8 high values of m × 28 n) = ~224 pairs: 6e5s. Way too slow.
# 
# Different angle: the analysis is FINE up to J=16, with λ_min ≈ -1e-13 at machine floor. The "instability" at J >= 20 is a feature of the test function — perhaps the Hermite-Gauss basis function gets to evaluate L-function "modes" that are below the numerical floor of FLOAT64.
# 
# However, looking at signed λ_min values for J>=20, they grow rapidly: -3.8e-10, -3.6e-7, -1.15e-4 for ζ at T0=30. 
# These are NOT just noise but a real numerical artifact growing with J.
# 
# The key insight: when computing M_zeros at large J, we compute psi_27(x) at x=(γ-T0) with |x|~10. 
# psi_27 has 27 oscillations and very large dynamic range. The recurrence loses precision exponentially with J.
# 
# Let me use mpmath for both M_zeros and M_arith for J>=20, working in high precision throughout.
# But mp for 3M primes × 28 (m,n) is prohibitive.
# 
# A practical compromise: compute Q via float128 (numpy longdouble) for the critical computations. 
# Or: use a precision-preserving algorithm for the prime sum.
# 
# Or maybe simplest: use mpmath to recompute Q ONLY for J=20, 24, 28 entries, at low dps (~30 digits).
# Use TRUNCATED prime sum for X=5e7 with mp arithmetic, but vectorize via numpy by promoting to numpy mp arrays — not great.
# 
# Let me try a different approach: use the Hermite function evaluation at high J in EXTENDED precision (using numpy float128). Check if scipy supports.

print("Test float128 / longdouble precision:")
print(f"numpy float128 eps: {np.finfo(np.longdouble).eps}")
print(f"numpy float64 eps: {np.finfo(np.float64).eps}")


Test float128 / longdouble precision:
numpy float128 eps: 1.0842021724855044e-19
numpy float64 eps: 2.220446049250313e-16


In [36]:

# float128 gives 19 digits vs 16 of float64. That's only ~3 extra digits. Not enough for J=28 where the error is ~1e-4.
# We'd need at least 25-30 digits.
# 
# Let me think about this differently. The trace residuals follow a clear pattern:
# trace_residual_rel goes from 1e-14 at J=4..16 to 1e-10 at J=20 to 1e-5 at J=28.
# 
# This is *not* the actual analysis residual after using a sufficient prime cutoff X=5e7. Rather it's a NUMERICAL precision issue from the Hermite-Gauss computation/Laguerre evaluation at high indices.
# 
# Key observation: the spec mentions "robust, high-precision zero-finding workflow ... at a working precision of at least 80 decimal digits" and "high-precision numbers as strings". So the project EXPECTS some mp arithmetic.
# 
# I will compute Q in HIGH PRECISION using mpmath for J >= 20 with a moderate dps (say 50). This will be slow but tractable.
# Plan: use mp directly for F0 evaluation and prime sum. But 3M primes × dps=50 in mp Python is ~30 microseconds per multiplication × 3M = 100s per (m,n) pair × 406 pairs = 40000s. Too slow.
# 
# Better plan: use mpmath ARRAYS via gmpy2 / flint to vectorize. Or factor the Laguerre evaluation:
# Compute Laguerre polynomial COEFFICIENTS once per (M, alpha), then evaluate via Horner at all v values.
# 
# Wait — actually the issue might just be in the Laguerre polynomial evaluation at large x. Let me check eval_genlaguerre stability.
# 
# scipy.special.eval_genlaguerre uses three-term recurrence which is forward stable for SOME parameter regimes but backward stable for others. At large x and high degree, can lose digits.
# 
# Let me use mpmath to compute the truth value at one problematic point and compare:
v = 17.7
M, alpha = 27, 0
lag_scipy = sps.eval_genlaguerre(M, alpha, v**2/2.0)
mp.mp.dps = 60
lag_mp = mp.laguerre(M, alpha, mp.mpf(v)**2 / 2)
print(f"scipy: {lag_scipy:.10e}")
print(f"mpmath: {float(lag_mp):.10e}")
print(f"Relative diff: {abs(float(lag_mp) - lag_scipy)/abs(float(lag_mp)):.2e}")


scipy: -5.1193141401e+28
mpmath: -5.1193141401e+28
Relative diff: 1.20e-15


In [37]:

# scipy is accurate to 1.2e-15 at this point. But then F0 = -5e28 × exp(-78.3) = -5e28 × 1.5e-34 = -7e-6.
# Final F0 has relative error inherited from L_27: ~1e-15. Multiplied by oscillating signs and summed ~3M times:
# Random walk error ~ sqrt(3M) × 1e-15 × typical_value (~1) = 1.7e-12.
# Hmm so theoretical limit should be ~1e-12, but we observe 1e-4. Something else is going on.
# 
# Let me check the SIGN of trace residual for ζ at T0=30, J=20-28:
# J=20: residual = +9.77e-10 (Mz > Marith by 1e-9)
# J=24: residual = +6.97e-7
# J=28: residual = +1.70e-4
# 
# These are large positive residuals. Could they reflect a missing contribution to M_arith?
# 
# At J=28, the test function phi_27 phi_27 has Fourier transform F0_{27,27}(v) = e^{-v^2/4} L_27(v^2/2)
# whose tail extends to v ~ 2*sqrt(2*27) = ~14. Beyond v=14, the function decays as e^{-v^2/4} × oscillations.
# v = k sigma log p = log p (k=1, sigma=1). So v=14 corresponds to log p = 14, i.e., p = e^14 ≈ 1.2 million.
# v_max for sigma=1, X=5e7 is log(5e7) = 17.7. So v from 0 to 17.7, but with most weight at LARGE v if F0 has its mass spread.
# 
# But F0_{27,27}(v) is concentrated near v=0 with a Gaussian envelope e^{-v^2/4}. At v=14, e^{-49} ~ 5e-22. 
# So contributions from p > e^14 are tiny.
# 
# BUT: F0_{27,27}(v) has polynomial degree 54 in v (L_27 contributes degree 54 in v). It oscillates within its envelope.
# So at intermediate v (say v=5..10), F0 magnitude can be moderate.
# 
# Let me look at F0_{27,27} envelope across v:
v_test = np.linspace(0.5, 17.7, 200)
f0_27 = F0_mn_vec(27, 27, v_test)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
ax.semilogy(v_test, np.abs(f0_27.real), label='|Re F0_{27,27}|')
ax.set_xlabel('v')
ax.set_ylabel('|F0_{27,27}(v)|')
ax.set_title('Magnitude of F0_{27,27}(v)')
ax.grid(True)
print(f"|F0_{{27,27}}| max in [0.5, 17.7]: {np.max(np.abs(f0_27.real)):.3e}, at v={v_test[np.argmax(np.abs(f0_27.real))]:.2f}")
print(f"|F0_{{27,27}}| at v=5: {abs(F0_mn_vec(27,27,np.array([5.0]))[0]):.3e}")
print(f"|F0_{{27,27}}| at v=10: {abs(F0_mn_vec(27,27,np.array([10.0]))[0]):.3e}")
print(f"|F0_{{27,27}}| at v=15: {abs(F0_mn_vec(27,27,np.array([15.0]))[0]):.3e}")


|F0_{27,27}| max in [0.5, 17.7]: 3.997e-01, at v=0.50
|F0_{27,27}| at v=5: 7.055e-02
|F0_{27,27}| at v=10: 2.644e-02
|F0_{27,27}| at v=15: 7.111e-02


In [38]:

# Wait, |F0| at v=15 is 0.07 (similar to v=5). That's surprising — should be exp(-225/4)=10^-25, but L_27((225/2))*exp(-56) might give something larger.
# Actually L_27(112.5) is a huge polynomial value (~10^25), times exp(-56) ~ 10^-25 — the product depends on cancellation.
# But our test says it's 0.07. Let me verify with mpmath:
mp.mp.dps = 60
def F0_mn_mp(m, n, v):
 M = min(m, n); N = max(m, n); d = N - M
 mp_v = mp.mpf(v)
 if d == 0:
 prefac = mp.mpf(1)
 else:
 log_fac = mp.log(mp.factorial(N)) - mp.log(mp.factorial(M))
 prefac = mp.exp(-log_fac/2)
 lag = mp.laguerre(M, d, mp_v**2 / 2)
 base = mp.exp(-mp_v**2/4) * prefac * lag
 if d == 0:
 return mp.mpc(base, 0)
 poly = (mp_v / mp.sqrt(2))**d
 minus_i_d = mp.mpc(0, -1)**d
 return base * poly * minus_i_d

for v in [0.5, 5.0, 10.0, 15.0, 17.7]:
 a = F0_mn_vec(27, 27, np.array([v]))[0]
 b = F0_mn_mp(27, 27, v)
 diff = abs(complex(a) - complex(b))
 print(f"v={v}: scipy={complex(a):.6e}, mpmath={complex(b):.6e}, |diff|={diff:.2e}, rel={diff/abs(b):.2e}")


TypeError: unsupported format string passed to mpf.__format__

In [39]:

for v in [0.5, 5.0, 10.0, 15.0, 17.7]:
 a = complex(F0_mn_vec(27, 27, np.array([v]))[0])
 b = complex(F0_mn_mp(27, 27, v))
 diff = abs(a - b)
 print(f"v={v}: scipy={a:.6e}, mpmath={b:.6e}, |diff|={diff:.2e}, rel={diff/abs(b):.2e}")


v=0.5: scipy=-3.996980e-01+0.000000e+00j, mpmath=-3.996980e-01+0.000000e+00j, |diff|=1.11e-16, rel=2.78e-16
v=5.0: scipy=-7.055221e-02+0.000000e+00j, mpmath=-7.055221e-02+0.000000e+00j, |diff|=5.55e-17, rel=7.87e-16
v=10.0: scipy=-2.643994e-02+0.000000e+00j, mpmath=-2.643994e-02+0.000000e+00j, |diff|=8.33e-17, rel=3.15e-15
v=15.0: scipy=-7.110942e-02+0.000000e+00j, mpmath=-7.110942e-02+0.000000e+00j, |diff|=5.55e-17, rel=7.81e-16
v=17.7: scipy=-4.945181e-06+0.000000e+00j, mpmath=-4.945181e-06+0.000000e+00j, |diff|=7.62e-21, rel=1.54e-15


In [40]:

# F0 itself is accurate to 1e-15. So the cancellation in the prime sum must be where precision is lost.
# Let me estimate the precision floor properly.
# Actually compute the sum at sigma=1, T0=30, J=28, m=n=27 prime contribution in mpmath at low precision:

# That's 3M iterations at high prec - too slow. Let me try a SUBSET of primes.
# Or estimate sigma_error from the sum of abs values:
import sys
mp.mp.dps = 30
T0 = 30.0
sigma = 1.0
m, n = 27, 27
# Take primes up to 1e6 (60K primes); evaluate F0 in mp arithmetic; sum

N_test = 100000 # first 100K primes
p_sub = primes[:N_test]
log_p_sub = np.log(p_sub.astype(np.float64))
weight_sub = -log_p_sub / np.sqrt(p_sub.astype(np.float64)) / np.pi
# k=1 only here
v_sub = sigma * log_p_sub

# Sum via float64
f0_arr_f64 = F0_mn_vec(m, n, v_sub)
phase = np.exp(-1j * log_p_sub * T0)
sum_f64 = np.sum(weight_sub * np.real(f0_arr_f64 * phase))
print(f"Float64 sum over first {N_test} primes: {sum_f64:.10e}")

# Use mpmath in batches for selected primes -- actually let me just trust the analysis.
# 
# Let me think about this differently. The precision floor should be ~1e-14 for a sum like sum a_i x_i with |a_i|~1e-1 and 3M terms with cancellation. Each summand has relative error 1e-15, sum has random walk error ~1e-15 × sqrt(3e6) × typical magnitude ~3e-13. So we should get ~1e-12, not 1e-4.
# 
# The actual issue must be in something else. Let me check whether F0 values at HIGH v have huge dynamic range.
# F0_{m,n}(v) for m, n with m+n=54 — the polynomial degree is 54. At intermediate v, values can swing by orders of magnitude.

f0_arr = F0_mn_vec(m, n, v_sub)
print(f"F0_{{27,27}}(v_sub) stats: min={np.min(np.abs(f0_arr)):.3e}, max={np.max(np.abs(f0_arr)):.3e}")
print(f"Sum |weight*F0|: {np.sum(np.abs(weight_sub * f0_arr)):.3e}")
print(f"|Sum weight*F0*phase|: {abs(np.sum(weight_sub * f0_arr * phase)):.3e}")
# Ratio = condition number


Float64 sum over first 100000 primes: 1.4937300850e+00
F0_{27,27}(v_sub) stats: min=1.352e-06, max=1.600e-01
Sum |weight*F0|: 5.251e+01
|Sum weight*F0*phase|: 1.523e+00


In [41]:

# Condition number ~50 — not catastrophic. So per-entry precision floor should be ~1e-15 × 50 = 5e-14, but the trace error is 1e-4 at J=28!
# 
# Let me actually try doing the sum at higher precision for ONE matrix element and see how it differs.

# Let me check trace error at J=20 separately. 
# Check: does trace(M_zeros) match (sum over n of <psi_n psi_n at zeros>)?
# This is independent computation, no Laguerre. Likely accurate.
# 
# Where is the precision loss? Let me check the explicit (m=n=27) entry vs mpmath truth using a smaller prime set.
N_test = 50000
p_sub = primes[:N_test].astype(np.float64)
log_p_sub = np.log(p_sub)
weight_sub = -log_p_sub / np.sqrt(p_sub) / np.pi
v_sub = 1.0 * log_p_sub
T0 = 30.0

# Float64
f0_arr_f64 = F0_mn_vec(27, 27, v_sub)
phase_f64 = np.exp(-1j * log_p_sub * T0)
sum_f64 = np.sum(weight_sub * np.real(f0_arr_f64 * phase_f64))

# Compute reference via mp -- 50K iterations, ~30s
mp.mp.dps = 30
t0 = time.time()
s_mp = mp.mpf(0)
for i in range(N_test):
 p = p_sub[i]
 lp = np.log(p) # use float64 log p, then convert (since prime is integer)
 mp_lp = mp.log(mp.mpf(int(primes[i]))) # exact log
 f0 = F0_mn_mp(27, 27, float(mp_lp))
 phase = mp.exp(mp.mpc(0, -float(mp_lp) * T0))
 w = -float(mp_lp) / mp.sqrt(mp.mpf(int(primes[i]))) / mp.pi
 s_mp += w * mp.re(f0 * phase)
elapsed = time.time() - t0
print(f"mp sum over {N_test} primes: {float(s_mp):.10e} in {elapsed:.1f}s")
print(f"f64 sum: {sum_f64:.10e}")
print(f"diff: {sum_f64 - float(s_mp):.3e}")


mp sum over 50000 primes: 4.8536980419e-01 in 6.4s
f64 sum: 4.8536980419e-01
diff: -1.110e-16


In [42]:

# Float64 vs mp agree to 1e-16 for 50K primes. So precision floor is fine here.
# The issue must accumulate over 3M primes. Let me check.

# Check: float64 sum over a larger range
N_test_large = 1000000
p_sub = primes[:N_test_large].astype(np.float64)
log_p_sub = np.log(p_sub)
weight_sub = -log_p_sub / np.sqrt(p_sub) / np.pi
v_sub = 1.0 * log_p_sub
T0 = 30.0

f0_arr = F0_mn_vec(27, 27, v_sub)
phase = np.exp(-1j * log_p_sub * T0)

# Standard sum
s1 = np.sum(weight_sub * np.real(f0_arr * phase))
# Kahan-compensated sum
contributions = weight_sub * np.real(f0_arr * phase)
s2 = np.float64(0); c = np.float64(0)
# Use built-in math.fsum for higher precision
import math
s_fsum = math.fsum(contributions.tolist())
print(f"np.sum: {s1:.15e}")
print(f"math.fsum: {s_fsum:.15e}")
print(f"diff: {s1 - s_fsum:.3e}")


np.sum: -8.060954108066354e-02
math.fsum: -8.060954108066323e-02
diff: -3.053e-16


In [43]:

# Diff between np.sum and exact-rounded math.fsum is just 3e-16. Precision is fine.
# So the original results for J=28 (residual ~1e-4) cannot be due to f64 sum precision.
# 
# Wait - perhaps my hermite_functions recurrence is unstable at high J for large |x|?
# Let me check. At J=27, x ranges from (γ - T0)/sigma where γ-T0 can be up to ~30 (we have zeros up to |γ-T0|<30).
# 
# Hermite recurrence: psi_{n+1}(x) = sqrt(2/(n+1)) x psi_n(x) - sqrt(n/(n+1)) psi_{n-1}(x)
# For large |x| (around sqrt(2n)), it's stable. For |x| > sqrt(2n), the recurrence amplifies errors.
# At J=27, sqrt(54) ≈ 7.3. So |x| > 7.3 is dangerous.
# For x up to 30, recurrence has ~3 amplification per step × 27 steps = 3^27 = 10^13 amplification of float64 error => 10^13 * 1e-16 = 1e-3 relative error in psi_27 at |x|=30.

# Let me verify:
x_check = np.array([5.0, 7.0, 10.0, 15.0, 20.0, 30.0])
Psi_check = hermite_functions(x_check, 28)
# Compare with mp
mp.mp.dps = 50
for xi in x_check:
 psi_28_recur = Psi_check[27, list(x_check).index(xi)]
 # Compute via mp
 def psi_n_mp(n, x_in):
 x = mp.mpf(x_in)
 p0 = mp.power(mp.pi, mp.mpf(-1)/4) * mp.exp(-x**2/2)
 if n == 0: return p0
 p1 = mp.sqrt(2) * x * p0
 if n == 1: return p1
 for k in range(1, n):
 p2 = mp.sqrt(mp.mpf(2)/(k+1)) * x * p1 - mp.sqrt(mp.mpf(k)/(k+1)) * p0
 p0, p1 = p1, p2
 return p1
 psi_27_mp = psi_n_mp(27, xi)
 print(f"x={xi}: psi_27 recur={psi_28_recur:.6e}, mp={float(psi_27_mp):.6e}, diff={abs(psi_28_recur - float(psi_27_mp)):.2e}")


x=5.0: psi_27 recur=-1.665846e-01, mp=-1.665846e-01, diff=1.39e-16
x=7.0: psi_27 recur=4.860409e-01, mp=4.860409e-01, diff=7.22e-16
x=10.0: psi_27 recur=2.085040e-06, mp=2.085040e-06, diff=1.27e-21
x=15.0: psi_27 recur=2.867429e-28, mp=2.867429e-28, diff=2.24e-43
x=20.0: psi_27 recur=9.841906e-63, mp=9.841906e-63, diff=0.00e+00
x=30.0: psi_27 recur=1.927379e-166, mp=1.927379e-166, diff=6.02e-182


In [44]:

# Hermite functions are accurate to machine precision. So that's not the issue.
# 
# Wait — I had mistakenly assumed the trace residual at high J is from precision. Let me look more carefully.
# Actually look at the relationship: ALL high-J pairs (zeta and Lchi) show similar pattern of trace_residual_rel and λ_min growing together at J=20-28.
# 
# Could it be that with the FIXED CACHE of L(chi) zeros (only 130 zeros, up to t=200), we have missing zeros that need to be summed?
# For ζ at T0=120, σ=1, we sum over zeros up to ~140. Need zeros up to T0+10σ=130 to be safe. We have zeros up to γ ≈ 141. So OK.
# For L(chi) at T0=120, similar — we have zeros up to 199. OK.
# But for J=28: psi_27 has its outermost peak near sqrt(2*27) ≈ 7.3, so the relevant zeros are within |γ-T0|<10 typically. With γ extending to 30 from T0, we should be safe.
# 
# Could the problem be MISSING zeros within the relevant window — i.e., the cached zero lists are incomplete?
# Lchi has 130 zeros in [0, 200] — average spacing 1.5. For T0=85.7, σ=1, J=28: relevant window ~[75, 96], expect 10-15 zeros there. We're using zeros with |γ-T0|<30, that's 55-115, plenty.
# 
# Actually wait - I keep zlist_relevant = [g for g in zeros_list if abs(float(g) - T0) < 30]. Then build Psi_full with J_max=28. Number of zeros used = 40-43 for L(chi). Should be enough.
# 
# Let me hypothesize: the arch matrix calculation may be insufficiently accurate at high J. Let me increase quadrature resolution and check.
m, n = 27, 27
T0 = 30.0
for npts in [4000, 8000, 16000, 32000]:
 a = archim_matrix_direct(T0, 1.0, 28, a=0, n_pts=npts, x_lim=20.0)
 print(f"npts={npts}: arch[27,27]={a[27,27]:.10e}, arch[26,27]={a[26,27]:.10e}")


npts=4000: arch[27,27]=2.4631280495e-01, arch[26,27]=1.9795909630e-02
npts=8000: arch[27,27]=2.4631280495e-01, arch[26,27]=1.9795909630e-02
npts=16000: arch[27,27]=2.4631280495e-01, arch[26,27]=1.9795909630e-02
npts=32000: arch[27,27]=2.4631280495e-01, arch[26,27]=1.9795909630e-02


In [45]:

# Arch matrix converged. Let me look at TRACE explicitly:
T0 = 30.0
J = 28
# tr(M_zeros) at sigma=1
x = np.array([(float(g) - T0)/1.0 for g in zeta_zeros if abs(float(g) - T0)<30], dtype=np.float64)
Psi_full = hermite_functions(x, 28)
Mz_28 = (Psi_full @ Psi_full.T) / 1.0
print(f"tr(M_zeros) = {np.trace(Mz_28):.10e}")

# tr(arch_28)
arch_28 = archim_matrix_direct(T0, 1.0, 28, a=0)
print(f"tr(arch) = {np.trace(arch_28):.10e}")

# tr(Mprime_28)
Mp_28 = Mprime_zeta_dict[30.0]
print(f"tr(Mp) = {np.trace(Mp_28):.10e}")
print(f"sum: {np.trace(Mp_28) + np.trace(arch_28):.10e}")
print(f"residual: {np.trace(Mz_28) - np.trace(Mp_28) - np.trace(arch_28):.3e}")
print()
print("Trace by index (cumulative diagonal):")
diag_Mz = np.diag(Mz_28)
diag_Marith = np.diag(Mp_28 + arch_28)
diag_diff = diag_Mz - diag_Marith
for i in range(28):
 print(f" n={i}: Mz_diag={diag_Mz[i]:.4e}, Marith_diag={diag_Marith[i]:.4e}, diff={diag_diff[i]:.3e}")


tr(M_zeros) = 6.3767453525e+00
tr(arch) = 6.9312534401e+00
tr(Mp) = -5.5467857762e-01
sum: 6.3765748625e+00
residual: 1.705e-04

Trace by index (cumulative diagonal):
 n=0: Mz_diag=4.7111e-01, Marith_diag=4.7111e-01, diff=-9.381e-15
 n=1: Mz_diag=1.7182e-01, Marith_diag=1.7182e-01, diff=-1.033e-14
 n=2: Mz_diag=1.0963e-01, Marith_diag=1.0963e-01, diff=-7.397e-15
 n=3: Mz_diag=2.5689e-01, Marith_diag=2.5689e-01, diff=-1.138e-14
 n=4: Mz_diag=1.8288e-01, Marith_diag=1.8288e-01, diff=-8.354e-15
 n=5: Mz_diag=4.7431e-01, Marith_diag=4.7431e-01, diff=-1.071e-14
 n=6: Mz_diag=2.8788e-01, Marith_diag=2.8788e-01, diff=-1.221e-14
 n=7: Mz_diag=2.8444e-01, Marith_diag=2.8444e-01, diff=-3.220e-15
 n=8: Mz_diag=7.1767e-03, Marith_diag=7.1767e-03, diff=-1.251e-14
 n=9: Mz_diag=2.4439e-01, Marith_diag=2.4439e-01, diff=-7.605e-15
 n=10: Mz_diag=2.2127e-01, Marith_diag=2.2127e-01, diff=-1.249e-14
 n=11: Mz_diag=2.1685e-01, Marith_diag=2.1685e-01, diff=-7.244e-15
 n=12: Mz_diag=1.7419e-01, Marith_diag=

In [46]:

# AHA! The diff grows EXPONENTIALLY in the diagonal as n increases beyond 15.
# n=14: 7e-14, n=20: 1e-9, n=27: 2e-4. Each step ~5x.
# 
# This is a precision loss in the prime sum at high (m=n). Maybe it's the Laguerre evaluation after all.
# Let me check: at m=n=27, scipy eval_genlaguerre had ~1e-15 relative error. But the typical magnitude of F0_{27,27}(v) at small v can be O(1), and the prime sum has 3M terms.
# 
# Random walk: variance accumulates as sum of variances per term. With 3M terms each contributing relative error 1e-15 × value, and typical values ~ weight*F0 ~ 0.1*0.3=0.03:
# stdev = sqrt(3M) × 1e-15 × 0.03 = 5e-12. But observed error is 2e-4. 8 orders of magnitude off.

# Let me check what scipy.special.eval_genlaguerre does for argument near (Hermite's biggest argument): v=17.7, x=v^2/2=156, M=27.
# The "transition" between L being O(1) magnitude and oscillatory polynomial is at x ~ 4*N for L_N. For N=27, transition at x=108. So x>108 is in oscillatory regime, where L_N alternates sign rapidly.
# scipy.eval_genlaguerre may use forward recurrence which is unstable in oscillatory regime.

# Let me check at large x:
for x in [50, 80, 100, 120, 140, 156, 160, 200]:
 s = sps.eval_genlaguerre(27, 0, float(x))
 mp.mp.dps = 50
 m_val = mp.laguerre(27, 0, mp.mpf(x))
 diff_rel = abs(s - float(m_val))/abs(float(m_val))
 print(f"x={x}: scipy={s:.6e}, mp={float(m_val):.6e}, rel diff={diff_rel:.2e}")


x=50: scipy=-1.903805e+09, mp=-1.903805e+09, rel diff=3.13e-15
x=80: scipy=-5.769182e+15, mp=-5.769182e+15, rel diff=2.43e-15
x=100: scipy=-7.081631e+20, mp=-7.081631e+20, rel diff=9.25e-16
x=120: scipy=-2.764597e+24, mp=-2.764597e+24, rel diff=3.88e-16
x=140: scipy=-9.655315e+26, mp=-9.655315e+26, rel diff=2.85e-16
x=156: scipy=-4.439083e+28, mp=-4.439083e+28, rel diff=7.93e-16
x=160: scipy=-1.061032e+29, mp=-1.061032e+29, rel diff=8.29e-16
x=200: scipy=-1.711797e+32, mp=-1.711797e+32, rel diff=2.10e-16


In [47]:

# scipy is fine. The issue is the PRODUCT L_27(x) * exp(-x/2) in F0.
# F0_27,27(v) = exp(-v^2/4) * L_27(v^2/2). For v=17.7: L_27(156)=-4e28, exp(-78.3)=1.4e-34, product = -5e-6.
# But float64 stores L_27 as approx 5e-6/1.4e-34 ≈ 5e28; relative precision 1e-15.
# Then F0 absolute error = 1e-15 × 5e28 × 1.4e-34 = 7e-21. Tiny.
# 
# Hmm. Let me check F0 directly across small v values which dominate:

# Wait — actually I built F0 in this order: lag * prefac * np.exp(-v^2/4). 
# At v=17.7: lag = -4e28, prefac = 1 (since m=n=27), exp(-78.3) = 1.4e-34. Product is ~-5.5e-6.
# So float64 arithmetic gives full-precision answer because we're multiplying a large number by a tiny number.

# Let me trace exactly where the precision loss for n=27 diagonal entry happens.
# Maybe it's the M_zeros calculation. We have only 13 zeros for ζ near T0=30. With psi_27 having some peaks at sqrt(2*27)=7.3 from center, the values at zeros are like 0.1-0.5. M_zeros[27,27] = sum_g psi_27((g-T0))^2 = 0.133.
# Each psi_27 evaluation has relative error 1e-15 × 0.5 = 5e-16. Sum of 13 squared values: error ~13 * 1e-16 = 1e-15.
# So M_zeros[27,27] should be 0.133 ± 1e-15. 
# 
# So M_arith[27,27] error must be ~2e-4. Where?
# 
# Let me compute Mp[27,27] via mp arithmetic for a SUBSET, then add the rest in f64. Or just bound the F0 sum errors.

# Strategy: identify if eval_genlaguerre at intermediate x has worse precision than I checked.
# In the relevant range v=log(p) for p in [2, 5e7], v in [0.69, 17.7]. x = v^2/2 in [0.24, 156.7].
# L_27 has 27 zeros distributed in (0, ~110). At those zeros, L_27 ~ 0 and the *relative* error becomes huge.
# But absolute error stays small. The contribution to the sum is small there.

# Maybe the issue is NOT precision but a missing/wrong term in M_arith at high J?
# Let me see what's the structure of the error: it grows like exp(c*n). This suggests a polynomial-times-Gaussian missing.
# 
# Hmm wait — could the polar terms matter at high J? Polar = h(i/2)+h(-i/2) for zeta.
# h(r) = phi_m(r) phi_n(r). At r=i/2: x = (i/2 - T0)/sigma = (i/2 - 30)/1 = -30 + i/2.
# psi_n(x) involves H_n(x) where x is large negative real with tiny imaginary part. exp(-x^2/2) = exp(-(30-i/2)^2/2) = exp(-(900 - 30i - 0.25)/2) = exp(-450 + 15i + 0.125).
# Modulus exp(-450). For n up to 27 H_n(x) can be ~|x|^n ~ 30^27 = 10^40. So |psi_n(i/2 from T0)|~ 1e40 * exp(-450) = 1e-156. Absolutely negligible.
# 
# Let me look at the full Mp matrix vs reference. Recompute Mprime in DOUBLE-DOUBLE arithmetic or longdouble. Or recompute in mpmath at low precision for a tiny subset of primes and see.

# Approach: compute the sum at higher precision using long double (numpy float128) for the accumulation.
# Try: split F0 contributions into chunks, sum with Kahan summation. But math.fsum was already exact.

# Let me check if the issue is in (-i v / sqrt(2))^(N-M) for off-diagonal entries:
# For m=n: prefac is 1 (no i power), so just real L_27 * exp(-v^2/4). Diagonal n=27 has issue 2e-4.
# So it's not the i power. Must be in L_27 * exp factor itself.

# Let me actually check the SUM contribution per prime BIN:
v_arr_local = log_p_arr # k=1 for k_arr==1, but k_arr varies
v_arr_local = sigma * k_arr * log_p_arr # sigma=1
f0 = F0_mn_vec(27, 27, v_arr_local)
coef = coef_zeta(p_arr, k_arr)
phase = np.exp(-1j * k_arr * log_p_arr * 30.0)
contribs = weight_arr * np.real(coef * phase * f0)

# Split by prime power
print("Contribution per k:")
for k in range(1, 8):
 mask = k_arr == k
 s = np.sum(contribs[mask])
 s_abs = np.sum(np.abs(contribs[mask]))
 print(f" k={k}, n_terms={mask.sum()}, sum={s:.6e}, sum_abs={s_abs:.6e}, cond={s_abs/(abs(s)+1e-30):.2e}")
# Total
print(f" TOTAL: {np.sum(contribs):.6e}")
print(f"Mp[27,27] from cache: {Mp_28[27,27]:.6e}")


Contribution per k:
 k=1, n_terms=3001134, sum=-8.659586e-02, sum_abs=7.870296e+01, cond=9.09e+02
 k=2, n_terms=908, sum=-4.136177e-02, sum_abs=1.461065e-01, cond=3.53e+00
 k=3, n_terms=73, sum=1.474717e-02, sum_abs=2.012608e-02, cond=1.36e+00
 k=4, n_terms=23, sum=-5.647344e-03, sum_abs=7.626293e-03, cond=1.35e+00
 k=5, n_terms=11, sum=5.336391e-03, sum_abs=6.718034e-03, cond=1.26e+00
 k=6, n_terms=8, sum=-4.364213e-04, sum_abs=2.024511e-03, cond=4.64e+00
 k=7, n_terms=5, sum=1.165896e-03, sum_abs=1.500015e-03, cond=1.29e+00
 TOTAL: -1.134266e-01
Mp[27,27] from cache: -1.134266e-01


In [48]:

# OK so the cancellation is ~1000:1 for k=1 sum. That's a more substantial loss but still:
# expected error = 1e-15 × 78.7 (sum of abs) = 7.9e-14. Actual error is ~2e-4. Still ~10 orders of magnitude off.
# 
# Maybe I'm wrong about the truth. Let me compute Mp[27,27] via mpmath at higher precision (slower but doable)
# Actually, let me just check by computing in float128:

# numpy doesn't broadcast eval_genlaguerre to float128 (scipy is f64 only). 
# Alternative: use mp.fdiv etc for batch ops via numpy.frompyfunc or list comprehension. Slow.

# Quick sanity test: compute F0_27,27 in mp arithmetic at a few specific v's where contribution is large:
mp.mp.dps = 30
# Find primes where |contribution| is large
contribs_abs = np.abs(contribs)
top_idx = np.argsort(contribs_abs)[-20:][::-1]
print("Top 20 contributions:")
mp_contribs = []
f64_contribs = []
for i in top_idx:
 p = int(primes[k_arr.cumsum()[i] if False else i] if False else None)
 # Simpler: extract directly
 # We need p and k
 # k_arr and p_arr were built by chained concat, but I have access to p_arr directly:
 pi = int(p_arr[i])
 ki = int(k_arr[i])
 lp = mp.log(mp.mpf(pi))
 v = mp.mpf(sigma) * ki * lp
 f0_mp_val = F0_mn_mp(27, 27, float(v))
 phase_mp = mp.exp(mp.mpc(0, -ki * float(lp) * 30.0))
 w = -float(lp) / mp.sqrt(mp.mpf(pi))**ki / mp.pi
 c_mp = w * mp.re(f0_mp_val * phase_mp)
 print(f" p={pi}, k={ki}: f64 contrib={contribs[i]:.6e}, mp contrib={float(c_mp):.6e}, diff={contribs[i]-float(c_mp):.2e}")


Top 20 contributions:


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

In [49]:

mp.mp.dps = 30
top_idx = np.argsort(np.abs(contribs))[-20:][::-1]
print("Top 20 contributions: idx, p, k, f64_contrib, mp_contrib, diff")
for i in top_idx:
 pi = int(p_arr[i])
 ki = int(k_arr[i])
 lp = mp.log(mp.mpf(pi))
 v = mp.mpf(sigma) * ki * lp
 f0_mp_val = F0_mn_mp(27, 27, float(v))
 phase_mp = mp.exp(mp.mpc(0, -ki * float(lp) * 30.0))
 w = -float(lp) / mp.sqrt(mp.mpf(pi))**ki / mp.pi
 c_mp = w * mp.re(f0_mp_val * phase_mp)
 print(f" i={i}, p={pi}, k={ki}: f64={contribs[i]:+.5e}, mp={float(c_mp):+.5e}, diff={contribs[i]-float(c_mp):+.2e}")


Top 20 contributions: idx, p, k, f64_contrib, mp_contrib, diff
 i=8, p=23, k=1: f64=+3.27369e-02, mp=+3.27369e-02, diff=-6.94e-18
 i=15, p=53, k=1: f64=+2.50393e-02, mp=+2.50393e-02, diff=-3.47e-18
 i=3001135, p=3, k=2: f64=-2.23161e-02, mp=-2.23161e-02, diff=+6.94e-18
 i=3001134, p=2, k=2: f64=-2.01336e-02, mp=-2.01336e-02, diff=+3.47e-18
 i=10, p=31, k=1: f64=+1.93622e-02, mp=+1.93622e-02, diff=+3.47e-18
 i=22, p=83, k=1: f64=-1.78943e-02, mp=-1.78943e-02, diff=-1.04e-17
 i=16, p=59, k=1: f64=-1.72344e-02, mp=-1.72344e-02, diff=-2.08e-17
 i=23, p=89, k=1: f64=+1.64291e-02, mp=+1.64291e-02, diff=+6.94e-18
 i=32, p=137, k=1: f64=-1.62649e-02, mp=-1.62649e-02, diff=+1.39e-17
 i=7, p=19, k=1: f64=+1.60147e-02, mp=+1.60147e-02, diff=-3.12e-17
 i=20, p=73, k=1: f64=+1.54453e-02, mp=+1.54453e-02, diff=+1.56e-17
 i=4, p=11, k=1: f64=-1.48575e-02, mp=-1.48575e-02, diff=+2.60e-17
 i=33, p=139, k=1: f64=-1.42122e-02, mp=-1.42122e-02, diff=-3.47e-18
 i=21, p=79, k=1: f64=-1.40324e-02, mp=-1.4032

In [50]:

# Individual contribs perfectly accurate at 1e-17. So no per-term issue.
# The sum then should be ~1e-15. But somehow Mp[27,27] computed by my code seems to mismatch the truth.
# 
# Maybe my mp eval of L_27 is wrong? Or the truth value of M_zeros[27,27] is wrong?
# Let me check M_zeros[27,27] independently in mp.

mp.mp.dps = 50
# zeta_zeros are mp.mpf
T0_test = 30.0
sigma_test = 1.0
z_relevant = [g for g in zeta_zeros if abs(float(g) - T0_test) < 30]
def psi_n_mp(n, x):
 p0 = mp.power(mp.pi, mp.mpf(-1)/4) * mp.exp(-x**2/2)
 if n == 0: return p0
 p1 = mp.sqrt(2) * x * p0
 if n == 1: return p1
 for k in range(1, n):
 p2 = mp.sqrt(mp.mpf(2)/(k+1)) * x * p1 - mp.sqrt(mp.mpf(k)/(k+1)) * p0
 p0, p1 = p1, p2
 return p1

mz_27_mp = mp.mpf(0)
for g in z_relevant:
 x = (g - T0_test)/sigma_test
 p = psi_n_mp(27, x)
 mz_27_mp += p**2
mz_27_mp = mz_27_mp / sigma_test
print(f"M_zeros[27,27] f64 = {Mz_28[27,27]:.15e}")
print(f"M_zeros[27,27] mp = {float(mz_27_mp):.15e}")
print(f"Diff: {Mz_28[27,27] - float(mz_27_mp):.2e}")


M_zeros[27,27] f64 = 1.331042333749858e-01
M_zeros[27,27] mp = 1.331042333749873e-01
Diff: -1.50e-15


In [51]:

# Mz[27,27] is accurate to 1.5e-15.
# Now Mp[27,27] - need to verify. Let me compute in mp at a moderate subset.
# Take first 100 primes and sum properly via mp; then add the rest in f64; compare.

# Actually the f64 sum over 3M primes gave -0.1134266e+00 and is reproducible (math.fsum within 1e-16).
# If Mz_28[27,27] - Mp[27,27] - arch_28[27,27] gives 2.18e-4, but Mz and arch are accurate, then Mp must be off by 2e-4.
# 
# Let me check: the SUM is computed correctly. So my MODEL for Mp is wrong by 2e-4! Either:
# (a) my Mp formula is missing a term that grows with J, or
# (b) the precise X-cutoff truncation error is much larger than expected for high-J test functions.
# 
# Hmm, (b) is interesting. The explicit formula formally requires summing over ALL primes (infinite). We truncate at X=5e7. The truncation error depends on how concentrated the test function is in frequency space.
# 
# For test function h_{mn}(r) = phi_m(r) phi_n(r), its Fourier transform F_{mn}(u) is concentrated near u=0 (since the function is in physical space, the FT decays with u). For high (m=n=27), F0_{27,27}(v) = exp(-v^2/4) L_27(v^2/2). 
# The function has support up to v ~ 2 sqrt(2*27) = 14.7 in classical sense. Beyond v=15, decays exponentially.
# 
# But the prime sum has u = k log p. For k=1 and X=5e7, max u = log(5e7) = 17.7. So we cover u in [log 2 ≈ 0.69, 17.7].
# For k=2: u in [2 log 2 ≈ 1.4, 2 log sqrt(5e7) ≈ 17.7]. Same.
# 
# So we DO sum across the relevant support. But by the truncation, we miss u > 17.7.
# For F0_{27,27}(v=17.7) ≈ 5e-6 from our earlier check. Tail contribution beyond v=17.7:
# Sum_{p > 5e7} (log p / sqrt(p)) F0_{27,27}(log p). For p ~ 5e7, log p / sqrt(p) ~ 18/7071 = 0.0025. F0 ~ 5e-6 at boundary, smaller beyond. Number of primes from 5e7 to 1e8 ~ 2.4M. Tail contribution ~ 2.4M × 0.0025 × |avg F0 with oscillations| ~ ... mostly cancels but can be order 1e-4 - 1e-3 from random walk if F0 doesn't decay fast.
# 
# Wait let me compute |F0_27,27| at large v more carefully. The Hermite-Gauss product has its mass within |v|< 2sqrt(2J).
# But L_27(v^2/2) at large v^2/2 actually grows like v^{2*27} / 27!. exp(-v^2/4) decays as exp(-v^2/4). Product: max at v near 2 sqrt(2*27)≈14.7.
# Beyond that, product decays like exp(-v^2/4 + 54 log v - log(27!)) which for v=20 gives exp(-100 + 54*3 - 28) = exp(34) ... wait that's huge.
# Hmm let me check numerically:
v_check = np.array([14.7, 17.7, 20.0, 25.0, 30.0])
f0_test = F0_mn_vec(27, 27, v_check)
print("|F0_27,27| at large v:")
for vi, f in zip(v_check, f0_test):
 print(f" v={vi}: F0={complex(f):.4e}, |F0|={abs(f):.4e}")


|F0_27,27| at large v:
 v=14.7: F0=-1.1124e-01+0.0000e+00j, |F0|=1.1124e-01
 v=17.7: F0=-4.9452e-06+0.0000e+00j, |F0|=4.9452e-06
 v=20.0: F0=-6.3680e-12+0.0000e+00j, |F0|=6.3680e-12
 v=25.0: F0=-2.2476e-30+0.0000e+00j, |F0|=2.2476e-30
 v=30.0: F0=-1.3611e-55+0.0000e+00j, |F0|=1.3611e-55


In [52]:

# OK F0_{27,27}(v) DOES decay rapidly beyond v=14.7 (where Hermite turns classically).
# At v=17.7, |F0|=5e-6. At v=20 (p ~ 5e8): 6e-12. Very small.
# 
# Tail contribution beyond X=5e7: sum_{p > 5e7} (log p / sqrt p) F0(log p). 
# log p ~ 17.7+, sqrt p ~ 7000+. (log p)/sqrt(p) ~ 0.0025. 
# F0(v=17.7)=5e-6, F0(v=18)=less. Per term: 0.0025 * 5e-6 = 1.3e-8 max.
# Number of primes from 5e7 to 1e8: ~2.4M; from 5e7 to ~e21 (limit where F0 still finite ~e-200): too far.
# 
# Actually F0(v=20)=6e-12, F0(v=25)=2e-30. So we only need primes up to about p = e^20 = 4.8e8 to be safe for J=28.
# Primes from 5e7 to 5e8: count ~24M (yes, 10x more than under 5e7). Each contributes ~ 0.0025 * |F0|.
# 
# Sum of |contributions| from these tail primes (in worst case):
# integral_5e7^5e8 (log p)/sqrt p * |F0(log p)| dπ(p) ≈ integral_log(5e7)^log(5e8) u/sqrt(e^u) * |F0(u)| * (e^u / u) du
# = integral u * (e^{u/2}) * F0(u) / u du = integral e^{u/2} F0(u) du
# For F0(u) ~ exp(-u^2/4) * polynomial-of-degree-54-in-u, we have e^{u/2} F0(u) decays still.
# At u=20: F0=6e-12, e^{u/2} = e^10 = 22000, product = 1.3e-7.
# Integrated over u from 17.7 to ~25: peak around u=18 maybe gives sum ~ 1e-3 ?
# 
# So truncation error from X=5e7 at J=28 may indeed be ~1e-3 to 1e-4 in M_arith_prime[27,27]. 
# This matches the observed trace_residual at J=28 of 2e-4 !!!
# 
# So X=5e7 is INSUFFICIENT for J=28 at σ=1. The hypothesis stated by the user fails for J>=20 at this X.
# Let me confirm with a simple test: extend the primes to X=2e8 (or more) for one configuration.

# Actually let me just look at the cumulative sum |F0| at increasing v to see truncation tail.
# We have v_arr_local from before: v = sigma * k * log_p, for sigma=1 most are < 17.7.

# Let me re-confirm the truncation hypothesis by computing F0(log p) for p in [5e7, ~5e8] using a sieve extension.
# Generate primes up to 5e8:
print("Extending prime sieve to 5e8 to estimate truncation tail...")

X2 = int(5e8)
# Memory: bool sieve 5e8 = 500MB. Tight but doable.
t0 = time.time()
sieve2 = np.ones(X2+1, dtype=bool)
sieve2[:2] = False
for i in range(2, int(X2**0.5)+1):
 if sieve2[i]:
 sieve2[i*i::i] = False
primes2 = np.nonzero(sieve2)[0].astype(np.int64)
del sieve2
print(f"Generated {len(primes2)} primes up to 5e8 in {time.time()-t0:.1f}s")
# Tail primes only
mask_tail = primes2 > 5e7
primes_tail = primes2[mask_tail]
print(f"Tail primes (5e7 < p <= 5e8): {len(primes_tail)}")


Extending prime sieve to 5e8 to estimate truncation tail...


Generated 26355867 primes up to 5e8 in 3.5s
Tail primes (5e7 < p <= 5e8): 23354733


In [53]:

# Compute the tail contribution to Mp[27,27] from p in (5e7, 5e8] (k=1 only):
log_p_tail = np.log(primes_tail.astype(np.float64))
weight_tail = -log_p_tail / np.sqrt(primes_tail.astype(np.float64)) / np.pi
T0 = 30.0
phase_tail = np.exp(-1j * log_p_tail * T0)

# F0_{27,27}(log p)
f0_tail = F0_mn_vec(27, 27, log_p_tail)
contribs_tail = weight_tail * np.real(f0_tail * phase_tail)
print(f"Tail (5e7,5e8] contribution to Mp[27,27]: {np.sum(contribs_tail):.6e}")
print(f"Sum of |contribs_tail|: {np.sum(np.abs(contribs_tail)):.6e}")
print(f"Original residual (Mz - Mp - arch)[27,27]: 2.18e-4")
# Free memory
del primes2, primes_tail, log_p_tail, weight_tail, phase_tail, f0_tail, contribs_tail
import gc; gc.collect()


Tail (5e7,5e8] contribution to Mp[27,27]: 2.180315e-04
Sum of |contribs_tail|: 1.259501e-03
Original residual (Mz - Mp - arch)[27,27]: 2.18e-4


969

In [54]:

# CONFIRMED! Tail (5e7, 5e8] contributes 2.18e-4 to Mp[27,27], EXACTLY accounting for the residual.
# 
# So the user's stated hypothesis "X=5×10⁷ is sufficient" is INCORRECT at high J (J >= 20 in our test).
# X=5e7 is sufficient for J <= 16 at σ=1, but fails for J >= 20.
# 
# This is a substantive scientific finding. To validate H1 properly at σ=1 across all J in {4,8,12,16,20,24,28},
# we need X = O(5e8) or larger.
# 
# Let me re-run the analysis with X=5e8 (10x larger). 24M primes, much more compute time.
# This is the correct answer. Let me proceed.
# 
# Use full sieve to 5e8: should fit in memory.
print("Building primes up to 5e8...")
t0 = time.time()
X_new = int(5e8)
sieve = np.ones(X_new+1, dtype=bool)
sieve[:2] = False
for i in range(2, int(X_new**0.5)+1):
 if sieve[i]:
 sieve[i*i::i] = False
primes_5e8 = np.nonzero(sieve)[0].astype(np.int64)
del sieve
print(f"Generated {len(primes_5e8)} primes up to 5e8 in {time.time()-t0:.1f}s")


Building primes up to 5e8...


Generated 26355867 primes up to 5e8 in 3.5s


In [55]:

# 26M primes. Building (p, k) arrays.
import gc
gc.collect()

# Free old (smaller) data
del p_arr, k_arr, log_p_arr, weight_arr, coef_arr_zeta, coef_arr_chi
del Mprime_zeta_dict, Mprime_chi_dict
gc.collect()

X_full = int(5e8)
p_arr, k_arr, log_p_arr, weight_arr = build_prime_kpow_arrays(primes_5e8, X_full)
print(f"Built prime-power arrays: {len(p_arr)} terms")
print(f"k counts: {np.bincount(k_arr)[1:][:10]}")


Built prime-power arrays: 26358615 terms
k counts: [26355867 2502 138 35 16 9 7 5
 4 4]


In [56]:

# 26M terms. Sum cost is 10x previous, so ~20 mins per (zeta, chi) for full prime-grid (5 T0 × 28×28/2 pairs)
# Actually it was 117s for 3M. So with 26M: ~117 × (26/3) = ~1000s = ~17 min per L-function.
# Plus we need to estimate if X=5e8 is sufficient... let me first check if even higher is needed.

# Quick check: tail contribution from (5e8, 5e9] for Mp[27,27] at T0=30. 
# But generating primes to 5e9 is too much memory (5GB sieve). Estimate via Mertens-like:
# F0_27,27 at v=20: 6e-12. At v=23 (p=10^10): F0 ~ e^{-23^2/4} L_27(23^2/2)... L_27(265): scaling like x^27/27! = 265^27/27! ~ 1e64/1.1e28 = 9e35. e^{-132} ~ 1e-58. Product = 9e35 * 1e-58 = 9e-23. So |F0|~1e-23. So (5e8, 5e9] tail: integral of 0.0025 * 1e-(some>10): negligible.
# Actually at v=20 (p~5e8), |F0|=6e-12. The tail from p > 5e8 contributes:
# integral_20^infty (log p)/sqrt(p) F0(log p) d (pi(p)) ~ integral_20^infty e^{u/2} F0(u) du
# F0(u) for u ≥ 20: ≤ 6e-12, so e^{u/2} F0(u) ≤ e^10 * 6e-12 = 1.3e-7 at u=20, decreasing.
# Integral from 20 to 25: < 7e-7. So tail (5e8, ~5e10]: max contribution ~ 7e-7 ≈ 1e-6. That's the truncation error.
# 
# This is comparable to what we want to verify. Maybe X=5e9 better. But sieve too big.
# 
# Compromise: use X=5e8 and accept residual ~1e-6 at J=28. If λ_min ~ 1e-6 at J=28, we'd say it's near the floor. 
# Better than X=5e7 where floor was 1e-4.

# Let me execute and report the trace residuals carefully.

print("Computing ζ prime-grid at X=5e8 (this will take ~15-20 min)...")
coef_arr_zeta = np.ones(len(p_arr), dtype=np.complex128)

t0 = time.time()
Mprime_zeta_dict = compute_prime_grid(p_arr, k_arr, log_p_arr, weight_arr, coef_arr_zeta, T0_list, sigma, J_max)
print(f"ζ prime grid (X=5e8) done in {time.time()-t0:.1f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [57]:

# Timed out. Let me parallelize or compute smaller. Since the previous X=5e7 finished in 117s for ζ, and X=5e8 is ~10x more, it should be ~1200s. But timeout is 900s. Need to break up.
# 
# Let me restart the cell and split by chunks. Actually let me see the state:
print("Current state - check if partial:")
print(f"Mprime_zeta_dict has T0 keys: {list(Mprime_zeta_dict.keys())[:5]}")
print(f"Sample matrix shape: {Mprime_zeta_dict[30.0].shape}")
print(f"Sum of |M[30.0]|: {np.sum(np.abs(Mprime_zeta_dict[30.0]))}")


Current state - check if partial:


NameError: name 'Mprime_zeta_dict' is not defined

In [58]:

# Kernel was reset by timeout. Reload state and split work into smaller chunks.
import os, pickle, time, gc, math
import numpy as np
import mpmath as mp
import scipy.special as sps
import pandas as pd

mp.mp.dps = 80

os.chdir('/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f')

# Load zeros
with open('zeta_zeros.pkl','rb') as f:
 zeta_zeros_str = pickle.load(f)
zeta_zeros = [mp.mpf(s) for s in zeta_zeros_str]

with open('Lchi_zeros.pkl','rb') as f:
 Lchi_pos_str = pickle.load(f)
with open('Lchi_zeros_neg.pkl','rb') as f:
 Lchi_neg_str = pickle.load(f)
Lchi_pos = [mp.mpf(s) for s in Lchi_pos_str]
Lchi_neg = [mp.mpf(s) for s in Lchi_neg_str]
Lchi_zeros_all = Lchi_pos + Lchi_neg

print(f"Loaded {len(zeta_zeros)} ζ zeros, {len(Lchi_zeros_all)} L(χ) zeros")


Loaded 49 ζ zeros, 260 L(χ) zeros


In [59]:

# Re-define helper functions
def hermite_functions(x, J):
 x = np.asarray(x, dtype=np.float64)
 psi = np.empty((J,) + x.shape, dtype=np.float64)
 psi[0] = np.pi**(-0.25) * np.exp(-x**2/2)
 if J > 1:
 psi[1] = np.sqrt(2.0) * x * psi[0]
 for n in range(1, J-1):
 psi[n+1] = np.sqrt(2.0/(n+1)) * x * psi[n] - np.sqrt(n/(n+1)) * psi[n-1]
 return psi

def F0_mn_vec(m, n, v_arr):
 M = min(m,n); N = max(m,n); d = N - M
 if d == 0:
 prefac_sqrt = 1.0
 else:
 log_fac = math.lgamma(N+1) - math.lgamma(M+1)
 prefac_sqrt = math.exp(-0.5 * log_fac)
 v2 = v_arr * v_arr
 lag = sps.eval_genlaguerre(M, d, v2/2.0)
 base = np.exp(-v2/4.0) * prefac_sqrt * lag
 if d == 0:
 return base.astype(np.complex128)
 poly = (v_arr / np.sqrt(2.0))**d
 minus_i_d = (-1j)**d
 return base * poly * minus_i_d

def M_zeros_matrix(zeros_list, T0, sigma, J):
 x = np.array([(float(g) - T0)/sigma for g in zeros_list], dtype=np.float64)
 Psi = hermite_functions(x, J)
 return (Psi @ Psi.T) / sigma

def archim_integrand_value(T0, sigma, a):
 from scipy.special import digamma
 if a == 0:
 const = -np.log(np.pi)
 else:
 const = np.log(5.0/np.pi)
 def f(x):
 t = T0 + sigma * x
 z = 1.0/4 + a/2.0 + 1j * t / 2.0
 return const + np.real(digamma(z))
 return f

def archim_matrix_direct(T0, sigma, J, a, n_pts=4000, x_lim=15.0):
 x = np.linspace(-x_lim, x_lim, n_pts+1)
 Psi = hermite_functions(x, J)
 dx = x[1]-x[0]
 f = archim_integrand_value(T0, sigma, a)
 g_vals = f(x)
 w = np.ones_like(x)
 w[1:-1:2] = 4
 w[2:-1:2] = 2
 w_arr_simp = w * dx / 3.0
 weighted = Psi * g_vals
 M = (Psi * w_arr_simp) @ (weighted.T) / (2*np.pi)
 return M

def chi_p_power(p_arr, k_arr):
 rem = (p_arr.astype(np.int64)) % 5
 chip = np.zeros(len(p_arr), dtype=np.complex128)
 chip[rem == 1] = 1
 chip[rem == 2] = 1j
 chip[rem == 3] = -1j
 chip[rem == 4] = -1
 return chip**k_arr

def build_prime_kpow_arrays(primes_arr, X):
 p_list = []
 k_list = []
 for k in range(1, 32):
 lim = X**(1.0/k)
 if k == 1:
 mask = primes_arr <= X
 else:
 mask = primes_arr <= lim + 1
 p_k = primes_arr[mask]
 if k > 1:
 p_k = p_k[p_k.astype(np.int64)**k <= X]
 if len(p_k) == 0:
 break
 p_list.append(p_k)
 k_list.append(np.full(len(p_k), k, dtype=np.int32))
 p_arr = np.concatenate(p_list).astype(np.float64)
 k_arr = np.concatenate(k_list)
 log_p = np.log(p_arr)
 p_to_kh = p_arr**(k_arr/2.0)
 weight = -log_p / p_to_kh / np.pi
 return p_arr, k_arr, log_p, weight

print("Helpers defined.")


Helpers defined.


In [60]:

# Build primes to 5e8
print("Building primes up to 5e8...")
t0 = time.time()
X_full = int(5e8)
sieve = np.ones(X_full+1, dtype=bool)
sieve[:2] = False
for i in range(2, int(X_full**0.5)+1):
 if sieve[i]:
 sieve[i*i::i] = False
primes_arr = np.nonzero(sieve)[0].astype(np.int64)
del sieve
gc.collect()
print(f"Generated {len(primes_arr)} primes up to {X_full} in {time.time()-t0:.1f}s")

p_arr, k_arr, log_p_arr, weight_arr = build_prime_kpow_arrays(primes_arr, X_full)
print(f"Prime-power terms: {len(p_arr)}")


Building primes up to 5e8...


Generated 26355867 primes up to 500000000 in 3.9s


Prime-power terms: 26358615


In [61]:

# Reformulate the prime-grid computation to be incremental, saving partials to disk so we don't lose work on timeout.

T0_list = [30.0, 46.13, 60.0, 85.7, 120.0]
J_max = 28
sigma = 1.0

cache_dir = '/workspace/87078ae8-1fa3-46f0-9985-b743eadf2a0f/Mprime_cache_5e8'
os.makedirs(cache_dir, exist_ok=True)

def compute_prime_grid_chunked(coef_arr, T0_list, sigma, J, cache_path):
 """Compute Mp[m,n] for all T0 and all (m,n). Save partial progress per (m,n) to disk."""
 # Check existing cache
 if os.path.exists(cache_path):
 with open(cache_path, 'rb') as f:
 state = pickle.load(f)
 M_dict = state['M_dict']
 done_pairs = state['done']
 print(f"Resuming from cache: {len(done_pairs)} (m,n) pairs done.")
 else:
 M_dict = {T0: np.zeros((J, J), dtype=np.float64) for T0 in T0_list}
 done_pairs = set()
 
 u_arr = k_arr * log_p_arr
 v_arr = sigma * u_arr
 phase_dict = {T0: np.exp(-1j * u_arr * T0) for T0 in T0_list}
 
 t_start = time.time()
 
 for m in range(J):
 for n in range(m, J):
 if (m, n) in done_pairs:
 continue
 f0 = F0_mn_vec(m, n, v_arr)
 base = coef_arr * f0 # complex
 for T0 in T0_list:
 phase = phase_dict[T0]
 val = float(np.sum(weight_arr * np.real(base * phase)))
 M_dict[T0][m, n] = val
 if m != n:
 M_dict[T0][n, m] = val
 done_pairs.add((m, n))
 elapsed = time.time() - t_start
 if (m * J + n) % 20 == 0 or (m, n) == (J-1, J-1):
 print(f" ({m},{n}) done, elapsed {elapsed:.1f}s, done={len(done_pairs)}/{J*(J+1)//2}")
 # Save state every 50 pairs
 if len(done_pairs) % 50 == 0:
 with open(cache_path, 'wb') as f:
 pickle.dump({'M_dict': M_dict, 'done': done_pairs}, f)
 
 # Final save
 with open(cache_path, 'wb') as f:
 pickle.dump({'M_dict': M_dict, 'done': done_pairs}, f)
 return M_dict

# ZETA: compute prime grid
print("Computing ζ prime-grid at X=5e8 (chunked, saving progress)...")
coef_arr_zeta = np.ones(len(p_arr), dtype=np.complex128)
zeta_cache_path = os.path.join(cache_dir, 'Mprime_zeta_X5e8.pkl')
t0 = time.time()
Mprime_zeta_dict = compute_prime_grid_chunked(coef_arr_zeta, T0_list, sigma, J_max, zeta_cache_path)
print(f"ζ prime grid complete in {time.time()-t0:.1f}s")


TimeoutError: Code execution timed out after 589.0 seconds